# Curating Metabolic Models

Curation is done in multiple steps: \
(1) Balancing \
(a) first manual balancing for mass imbalances \
(b) automatically overwriting metabolite and reaction information with BIGG information \
(c) second manual balancing for charge imbalances

(2) Duplicates
(a) automated based on Macaw
(b) manual

(3) Gap-Filling

(4) Manual fixes (Menoquinol, Ubiquinol, Sucrose)

!! Ready to use models are provided (see GitHub), if you want to dive into our analyses, you can directly skip to Notebook 03

## Imports
Macaw can be installed via pip \
pip install git+https://github.com/Devlin-Moyer/macaw.git@main

In [3]:
# basics
import os
import csv
import ast
import copy

# cobra & macaw
from cobra.io import read_sbml_model, write_sbml_model
from cobra.manipulation.validate import check_mass_balance
from cobra import Metabolite

from macaw.main import *

# advanced functions
from collections import defaultdict, Counter
from functools import reduce
from itertools import chain
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# my own functions
from functions import *

# PART 1

# Mass Balance Metabolic Models
This is the first manual mass balancing and in the following scripts written by Frowin Reichhardt from the AG Töpfer/University of Cologne are used to fix imbalanced reactions that he already detected in his own models.

## Paths

In [30]:
# Path to the model files
model_path = "../Models/01_carveme/"

# Path to the directory where you want to save the results aka mass balanced models
curation_path = "../Models/02_afterCuration/"

## Functions to check reactions for imbalances

In [31]:
# Creates a list of all internal reactions in the model
def internal_reactions(model):

    # Get all reaction_ids in the model
    model_reactions = []
    for reaction in model.reactions:
        model_reactions.append(reaction.id)

    # Filter out all exchanges from the list of reactions
    indicator_exchanges = ['EX','sink','Growth']
    internal_reactions = list(filter(lambda i: all(indicator not in i for indicator in indicator_exchanges), model_reactions))

    return internal_reactions

In [32]:
# Confirms if a single reaction is mass balanced
# This function checks if the sum of the elements in the reactants equals the sum of the elements in the products
# It returns True if the reaction is mass balanced, and False otherwise
def is_mass_balanced(reaction):

    # Create a dict to store the sum of each element in the reaction
    balance = defaultdict(int)

    # Iterate over the metabolites in the reaction
    for metabolite, coefficient in model.reactions.get_by_id(reaction).metabolites.items():

        # Check if the metabolite has elements
        if metabolite.elements is None or len(metabolite.elements) == 0:
            return False

        # Sum the elements in the balance dict using their coefficients
        for element, amount in (metabolite.elements).items():
            balance[element] += coefficient * amount

    # Returns True if all elements are balanced (in a margin of 1e-10)
    return all(-1e-10 < amount < 1e-10 for amount in balance.values())

In [33]:
# This function checks all internal reactions in the model and returns a list of unbalanced reactions
def find_mass_unbalanced_reactions(model):

    # Get all internal reactions in the model
    reactions = internal_reactions(model)

    # Check each reaction for mass balance using the is_mass_balanced function
    # and return a list of unbalanced reactions
    return [rxn for rxn in reactions if not is_mass_balanced(rxn)]

In [34]:
# This function combines the list of all imbalanced reactions from all models
# and create a DataFrame showing how often each imbalanced reaction occurs throughout the models
def reaction_abundance(imbalanced_models):
    # Create a list to collect all imbalanced reactions
    # and a dict to store the models containing these reactions
    unique_imbalanced_reactions = []
    occurences_reactions = {}
    # Create the unique list of imbalanced reactions by iterating through all reaction list in the dict and appending reactions that are not already in the list
    for reaction_list in imbalanced_models.values():
        for reaction in reaction_list:
            if reaction not in unique_imbalanced_reactions:
                unique_imbalanced_reactions.append(reaction)
    # Iterate through the list and find all models containing this reaction
    for reaction in unique_imbalanced_reactions:
        occurences_reactions[reaction] = []
        for imbalanced_model in imbalanced_models.keys():
            if reaction in imbalanced_models[imbalanced_model]:
                occurences_reactions[reaction].append(imbalanced_model)
    # Create a table with all results
    imbalanced_reactions = pd.DataFrame({
    "Reaction": occurences_reactions.keys(),
    "Occurences": [len(v) for v in occurences_reactions.values()],
    "Model IDs": occurences_reactions.values()})
    return imbalanced_reactions

## Functions to fix imbalanced reactions

In [35]:
def fix_cmcbtf(model):

    # Change the formula and charge of fcmcbtt_c in all models
    model.metabolites.get_by_id('fcmcbtt_c').formula = 'C33FeH48N5O13'
    model.metabolites.get_by_id('fcmcbtt_c').charge = 3

    # Change the charge of fcmcbtt_p and _e if they are present in the model
    if 'fcmcbtt_p' in model.metabolites:
        model.metabolites.get_by_id('fcmcbtt_p').charge = 3
    if 'fcmcbtt_e' in model.metabolites:
        model.metabolites.get_by_id('fcmcbtt_e').charge = 3

    # Subtract protons from the reactions CMCBTFR1 and CMCBTFU if they are present in the model
    if 'CMCBTFR1' in model.reactions:
        model.reactions.get_by_id('CMCBTFR1').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if 'CMCBTFR2' in model.reactions:
        model.reactions.get_by_id('CMCBTFR2').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if 'CMCBTFU' in model.reactions:
        model.reactions.get_by_id('CMCBTFU').subtract_metabolites({model.metabolites.get_by_id('h_c'): 1})

    return model

In [36]:
def fix_dhbsz3feabcpp(model):

    # Change the formula and charge of fe3dhbzs3_c
    model.metabolites.get_by_id('fe3dhbzs3_c').charge = 3
    model.metabolites.get_by_id('fe3dhbzs3_c').formula = 'C30FeH28N3O16'

    # Change the charge of fe3dhbzs3_p and _e
    if 'fe3dhbzs3_p' in model.metabolites:
        model.metabolites.get_by_id('fe3dhbzs3_p').charge = 3
        model.metabolites.get_by_id('fe3dhbzs3_e').charge = 3

    if 'feenter_c' in model.metabolites:
        model.metabolites.get_by_id('feenter_c').charge = 3

    # Subtract protons from the right side of reaction FE3DHBZS3R, FEDHBZS3R1 and FEDHBZS3R2
    if "FE3DHBZS3R" in model.reactions:
        model.reactions.get_by_id('FE3DHBZS3R').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if "FEDHBZS3R1" in model.reactions:
        model.reactions.get_by_id('FEDHBZS3R1').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if "FEDHBZS3R2" in model.reactions:
        model.reactions.get_by_id('FEDHBZS3R2').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if "FEDHBZS3R3" in model.reactions:
        model.reactions.get_by_id('FEDHBZS3R3').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if "FEENTERES" in model.reactions:
        model.reactions.get_by_id('FEENTERES').subtract_metabolites({model.metabolites.get_by_id('h_c'): -1})

    # Return the model
    return model

In [37]:
def fix_man6gpts(model):

    # Deleting objects
    # This wil delete all listed reactions and due to the 'True' argument also all resulting orphan metabolites. In this case only 'man6pglyc_e'
    model.remove_reactions(['MAN6Gpts', 'EX_man6pglyc_e'], True)

    # Create objects to be added to the model

    # Metabolites:

    # Metabolite manglyc_p
    manglyc_p = Metabolite(
        'manglyc_p',
        formula='C9H15O9',
        name='2(alpha-D-Mannosyl)-D-glycerate',
        compartment='p',
        charge =-1)

    # Metabolite manglyc_e
    manglyc_e = Metabolite(
        'manglyc_e',
        formula='C9H15O9',
        name='2(alpha-D-Mannosyl)-D-glycerate',
        compartment='e',
        charge =-1)

    # Reactions:

    # Reaction MANGLYCptspp
    MANGLYCptspp = Reaction('MANGLYCptspp')
    MANGLYCptspp.name = '2-O-alpha-mannosyl-D-glycerate transport via PEP:Pyr PTS (periplasm)'
    MANGLYCptspp.subsystem = ''
    MANGLYCptspp.lower_bound = 0.0
    MANGLYCptspp.upper_bound = 1000.0
    MANGLYCptspp.add_metabolites({
        manglyc_p: -1,
        model.metabolites.get_by_id('pep_c'): -1,
        model.metabolites.get_by_id('man6pglyc_c'): 1,
        model.metabolites.get_by_id('pyr_c'): 1
    })

    # Reaction MANGLYCtex
    MANGLYCtex = Reaction('MANGLYCtex')
    MANGLYCtex.name = '2-O-alpha-mannosyl-D-glycerate transport via diffusion (extracellular to periplasm)'
    MANGLYCtex.subsystem = ''
    MANGLYCtex.lower_bound = -1000.0
    MANGLYCtex.upper_bound = 1000.0
    MANGLYCtex.add_metabolites({
        manglyc_e: -1,
        manglyc_p: 1
    })

    # Reaction EX_manglyc_e
    EX_manglyc_e = Reaction('EX_manglyc_e')
    EX_manglyc_e.name = '2(alpha-D-Mannosyl)-D-glycerate exchange'
    EX_manglyc_e.subsystem = ''
    EX_manglyc_e.lower_bound = 0.0
    EX_manglyc_e.upper_bound = 1000.0
    EX_manglyc_e.add_metabolites({
        manglyc_e: -1
    })

    # Add reactions to the model
    model.add_reactions([MANGLYCptspp, MANGLYCtex, EX_manglyc_e])

    # Apply changes
    model.metabolites.get_by_id('man6pglyc_c').charge = -3
    model.metabolites.get_by_id('man6p_c').charge = -2

    # Return the modified model
    return model

In [38]:
def fix_mcbtfabcpp(model):

    # Change the formula and charge of fe3mcbtt_c
    model.metabolites.get_by_id('fe3mcbtt_c').charge = 3
    model.metabolites.get_by_id('fe3mcbtt_c').formula = 'C47FeH77N5O10'

    # Change the charge of fe3mcbtt_p and _e
    model.metabolites.get_by_id('fe3mcbtt_p').charge = 3
    model.metabolites.get_by_id('fe3mcbtt_e').charge = 3

    # Subtract protons from the right side of reaction MCBTFR1
    model.reactions.get_by_id('MCBTFR1').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})

    return model

In [39]:
def fix_salchs4feabcpp(model):

    # Change the formula and charge of salchs4fe_c
    model.metabolites.get_by_id('salchs4fe_c').charge = 3
    model.metabolites.get_by_id('salchs4fe_c').formula = 'C42FeH46N3O25'

    # Change the charge of salchs4fe_p and _e
    if 'salchs4fe_p' in model.metabolites:
        model.metabolites.get_by_id('salchs4fe_p').charge = 3
        model.metabolites.get_by_id('salchs4fe_e').charge = 3

    # Subtract protons from the right side of reaction SALCHS4FER1, SALCHS4FER2 and SALCHS4FER3
    if "SALCHS4FER1" in model.reactions:
        model.reactions.get_by_id('SALCHS4FER1').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if "SALCHS4FER2" in model.reactions:
        model.reactions.get_by_id('SALCHS4FER2').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})
    if "SALCHS4FER3" in model.reactions:
        model.reactions.get_by_id('SALCHS4FER3').subtract_metabolites({model.metabolites.get_by_id('h_c'): 2})

    return model

In [40]:
def fix_uaccpts(model):

    # Change the formula and charge of uaccg_e
    model.metabolites.get_by_id('uaccg_e').charge = 0
    model.metabolites.get_by_id('uaccg_e').formula = 'C20H29N3O19P2'

    # Change the charge of uamr_c
    model.metabolites.get_by_id('uamr_c').charge = -3

    return model

In [41]:
def fix_tagabc(model):

    # Change the formula and charge of tagur_e
    model.metabolites.get_by_id('tagur_e').formula = 'C6H10O7'

    # Change the charge of tagur_c
    model.metabolites.get_by_id('tagur_c').charge = -1

    return model

## Main

If a reaction is imbalanced in a model the corresponding function will be applied to it.
Afterwards the model gets saved as "modified_{model_id}" and a dictionary containing all modified_models as keys and their still imbalanced reactions (if existing) gets created

In [42]:
imbalanced_models = {}

for m in os.listdir(model_path):
    if m.endswith(".xml"):
        print(m)
        model = read_sbml_model(model_path+f'/{m}')
        if len(find_mass_unbalanced_reactions(model)) != 0:
            imbalanced_models[m] = find_mass_unbalanced_reactions(model)

AA6.xml
AA4.xml
AA5.xml
AA3.xml
AA1.xml
AA2.xml
AA7.xml


In [43]:
imbalanced_reactions = reaction_abundance(imbalanced_models)

In [44]:
imbalanced_reactions

,Reaction,Occurences,Model IDs
0,CMCBTFL,7,"[AA6.xml, AA4.xml, AA5.xml, AA3.xml, AA1.xml, ..."
1,SALCHS4FEabcpp,7,"[AA6.xml, AA4.xml, AA5.xml, AA3.xml, AA1.xml, ..."
2,CMCBTFabcpp,2,"[AA4.xml, AA7.xml]"
3,DHBSZ3FEabcpp,2,"[AA4.xml, AA3.xml]"
4,TAGabc,1,[AA5.xml]


In [45]:
reaction_dict = {}

# Iterate through all rows of the imbalanced_reactions DataFrame
for index, row in imbalanced_reactions.iterrows():
    reaction_dict[row[0]] = row[2]

# Combine CMCBTFabcpp and CMCBTFL into one entry since the function for these is the same
reaction_dict['CMCBTF'] = reaction_dict['CMCBTFabcpp'] + list(set(reaction_dict['CMCBTFL']) - set(reaction_dict['CMCBTFabcpp']))

/tmp/ipykernel_7991/2981178403.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  reaction_dict[row[0]] = row[2]


In [46]:
# Dict to save the models with imbalanced reactions after the modification by the Functions to fix imbalanced reactions
imbalanced_models_aftermod = {}

In [48]:
for m in os.listdir(model_path):
    if m.endswith(".xml"):

        print("checking", m)

        model = read_sbml_model(model_path + m)

        try:
            if m in reaction_dict["SALCHS4FEabcpp"]:
                model = fix_salchs4feabcpp(model)
        except KeyError:
            pass
        try:
            if m in reaction_dict["MCBTFabcpp"]:
                model = fix_mcbtfabcpp(model)
        except KeyError:
            pass
        try:
            if m in reaction_dict["CMCBTF"]:
                model = fix_cmcbtf(model)
        except KeyError:
            pass
        try:
            if m in reaction_dict["DHBSZ3FEabcpp"]:
                model = fix_dhbsz3feabcpp(model)
        except KeyError:
            pass
        try:
            if m in reaction_dict["MAN6Gpts"]:
                model = fix_man6gpts(model)
        except KeyError:
            pass
        try:
            if m in reaction_dict["UACCpts"]:
                model = fix_uaccpts(model)
        except KeyError:
            pass
        try:
            if m in reaction_dict["TAGabc"]:
                model = fix_tagabc(model)
        except KeyError:
            pass

        # Test if the fixes worked
        imbalanced_reactions_aftermod = find_mass_unbalanced_reactions(model)
        # print(imbalanced_reactions_aftermod)
        if len(imbalanced_reactions_aftermod) != 0:
            imbalanced_models_aftermod[m] = imbalanced_reactions_aftermod
        else:
            print("all fixed")
            path = f"{curation_path}{m}"
            write_sbml_model(model, path)

checking AA6.xml
all fixed
checking AA4.xml
all fixed
checking AA5.xml
all fixed
checking AA3.xml
all fixed
checking AA1.xml
all fixed
checking AA2.xml
all fixed
checking AA7.xml
all fixed


In [49]:
# To test if there are still models with imbalanced reactions. If this outputs only {}, then all models are mass balanced.
print(imbalanced_models_aftermod)

{}


# Charge Balance Metabolic Models with BIGG
With Frowins Scripts I was able to obtain stoichiometric consistent models; next step to balance the charges.
I use the BIGG database to compare the info there with model info and automatically replace charge and formula of metabolites with BIGG info if it seems more reasonable.
After this automated step, the remaining unbalanced reactions need to be manually curated.

## Load models

In [50]:
# import models after mass-balancing
models = {}
for model_name in (f for f in os.listdir(curation_path) if f.endswith(".xml")):
    model = read_sbml_model(f"{curation_path}{model_name}")
    model.solver = "cplex"
    models[model_name[:3]] = model  # it takes first three letters of the xml filename as the model name

models = {key: models[key] for key in sorted(models.keys())}  # sorts the dictionary alphabetically (AA1...AA7)

# this is specifically for my 7 models to be able to access them more easily; all models are stored in the "model" dict and can be accessed e.g. through models["AA1"] depending what you set as model name
AA1, AA2, AA3, AA4, AA5, AA6, AA7 = [models[f"AA{i}"] for i in range(1, 8)]

## Functions

In [51]:
def get_objective_value(model):
    print(f"value of objective for {model} is {model.optimize().objective_value}")

In [52]:
# checks the mass and charge balance for every reaction in a model
def check_balance(model, print_results=True):
    unbalanced_reactions = check_mass_balance(model)
    if print_results:
        print("There are {0} unbalanced reactions in {1}".format(len(unbalanced_reactions), model) )
    return unbalanced_reactions

In [53]:
# returns a pandas dataframe with metabolite info for a specific cobra model that includes: bigg_id, model_id, formula and charge
def extract_met_info_model(model):
    met_infos = []

    for met in model.metabolites:
        met_infos.append({
            "bigg_id": met.id.rsplit("_", 1)[0],  # strip compartment so that it matches the actual BIGG ID that also doesn't have compartments (e.g., glc__D_c to glc__D)
            "model_id": met.id,
            "model_formula": met.formula,
            "model_charge": met.charge
        })

    met_infos = pd.DataFrame(met_infos)
    return met_infos

In [54]:
# returns pandas dataframe with metabolite info from the model and from big and compares info about formula and charge state
def compare_bigg_modelMets(model_mets, list_unbalanced_mets):
    # Merge on BiGG ID
    merged = model_mets.merge(df_bigg_met, on="bigg_id", how="left")

    merged["charge_match"] = merged.apply(
        lambda row: row["model_charge"] in row["charges"] if isinstance(row["charges"], list) else False,
        axis=1
    )

    merged["formula_match"] = merged.apply(
        lambda row: row["model_formula"] in row["formulas"] if isinstance(row["formulas"], (list, set)) else False, axis=1
    )

    # adds another column to check if the metabolites are part of an unbalanced reaction (false = not part of unbalanced reactions, true = part of unbalanced reaction(s))
    merged['unbalanced'] = merged['model_id'].isin(list_unbalanced_mets)
    # merged['unbalanced'] = merged['model_id'].isin(list_unbalanced_mets).astype(int) (instead of true/false with 1/0)

    return merged

In [55]:
# filters the merged df to only show the rows (aka metabolites) where model info and bigg info do NOT match
def get_mismatches_after_merge(df_merge):
    mismatches = df_merge.loc[(df_merge['formula_match'] == False) | (df_merge['charge_match'] == False)]
    mismatches = mismatches[["model_id", "bigg_id", "model_charge", "charges", "model_formula", "formulas", "charge_match", "formula_match", "unbalanced"]]

    return mismatches

In [56]:
# returns a confusion matrix showing how much mismatching info about charge state and/or formula there is
# this function either takes one df from one model as an input or the merged dict where all models are saved
def get_confmat_charge_formula(df_merge):
    if isinstance(df_merge, pd.DataFrame):
        conf_matrix = df_merge.groupby(["charge_match", "formula_match"]).size().reset_index(name='count')
        print(conf_matrix)
    elif isinstance(df_merge, dict):
        conf_matrix = {
            "charge_match": ["False", "False", "True", "True"],
            "forumla_match": ["False", "True", "False", "True"],
        }
        conf_matrix = pd.DataFrame(conf_matrix)
        for i, item in enumerate(df_merge.values()):
            conf_matrix_model = item.groupby(["charge_match", "formula_match"]).size().reset_index(name='count')
            name = f"AA{i+1}"
            conf_matrix.insert(i+2, name, conf_matrix_model["count"])
        print(conf_matrix)

## Evaluate current state of models regarding charge balance

In [58]:
# dictionary to store unbalanced reactions
unbalanced_reactions_dict = {}

# models is the dict where all models are stored that were "imported" witch read_sbml_file()
for name, model in models.items():
    unbalanced_reactions = check_balance(model)
    unbalanced_reactions_dict[name] = unbalanced_reactions

# these numbers are in accordance with the numbers for "charge balance" reactions in the memote report
# when i do memote with the mass balanced models; directly after carveme we have more charge unbalanced reactions but the mass balancing already fixed some

There are 448 unbalanced reactions in AA1
There are 433 unbalanced reactions in AA2
There are 398 unbalanced reactions in AA3
There are 401 unbalanced reactions in AA4
There are 384 unbalanced reactions in AA5
There are 440 unbalanced reactions in AA6
There are 454 unbalanced reactions in AA7


In [59]:
# We know how many unbalanced reactions each model has on their own but what is the overlap?
unique_reactions = set()

# Loop through all models and collect reaction IDs
for model_name, unbalanced_reactions in unbalanced_reactions_dict.items():
    # Add the reaction ID to the set (sets are by default like 'Mengen', i.e. they only have unique elements)
    unique_reactions.update(reaction.id for reaction in unbalanced_reactions.keys())

# this is a list of all the reaction IDs that are charge unbalanced throughout all models
unique_reaction_ids = list(unique_reactions)

print("There are {0} charge unbalanced reactions throughout all models.".format(len(unique_reaction_ids)))
# print(unique_reaction_ids)

There are 816 charge unbalanced reactions throughout all models.


In [60]:
# we now the unbalanced reactions but which metabolites are part of these?
# go through all unbalanced (unique) reactions and get all participating metabolites
metabolite_counter_compartment = Counter()
metabolite_counter_name = Counter()
seen_reactions = set()  # Track reactions that were already counted

for model in models.values():
    for rxn_id in unique_reaction_ids:
        if rxn_id in model.reactions and rxn_id not in seen_reactions:
            reaction = model.reactions.get_by_id(rxn_id)
            for metabolite in reaction.metabolites:
                metabolite_counter_compartment[metabolite.id] += 1  # this is compartment specific, e.g. h2o_c and h2o_p are different metabolites
                metabolite_counter_name[metabolite.name] += 1  # h2o is only counted once not dependent on compartment
            seen_reactions.add(rxn_id)  # Mark this reaction as counted

In [61]:
pd.DataFrame(metabolite_counter_compartment.items())

,0,1
0,h2o_p,55
1,pgp140_p,1
2,pg140_p,2
3,pi_p,15
4,12dgr181_c,2
...,...,...
986,enter_e,1
987,feenter_e,1
988,bz_c,1
989,gdpmanur_c,2


In [62]:
# these are the amounts of unique metabolites that are part of unbalanced reactions

# compartment specific, e.g. h20_c and h2o_p are counted separately
print(len(metabolite_counter_compartment))
# h2o only exists once
print(len(metabolite_counter_name))

991
870


In [63]:
# get the amount of metabolites that are only part of v reactions or check which metabolites are part of the most reactions
filtered = {m: v for m, v in metabolite_counter_compartment.items() if v == 1}
print(len(filtered))

print(metabolite_counter_compartment.most_common(5))


426
[('h_c', 415), ('h2o_c', 263), ('atp_c', 139), ('coa_c', 124), ('ppi_c', 106)]


In [64]:
%matplotlib notebook

# Count how many keys have each count (i.e. histogram of values)
count_distribution = Counter(metabolite_counter_compartment.values())

# Plot
plt.bar(count_distribution.keys(), count_distribution.values())
plt.xlabel('Amount of Reactions a Metabolite is Part of')
plt.ylabel('Number of Metabolites with that count')
plt.title('Distribution of Metabolite occurrences in unbalanced reactions')
plt.show()


<IPython.core.display.Javascript object>

In [65]:
# the number show how often one metabolite is part of an unbalanced reaction
metabolite_counter_compartment

Counter({'h_c': 415,
         'h2o_c': 263,
         'atp_c': 139,
         'coa_c': 124,
         'ppi_c': 106,
         'pi_c': 89,
         'amp_c': 88,
         'adp_c': 58,
         'h2o_p': 55,
         'h_p': 49,
         'co2_c': 48,
         'nadh_c': 42,
         'ACP_c': 41,
         'nad_c': 40,
         'pyr_c': 39,
         'nadph_c': 38,
         'nadp_c': 35,
         'fad_c': 34,
         'fadh2_c': 34,
         'cmp_c': 26,
         'o2_c': 25,
         'glyc3p_c': 22,
         'g3p_c': 21,
         'glu__L_c': 20,
         'fe2_c': 19,
         'pep_c': 17,
         'pi_p': 15,
         'nh4_c': 15,
         'r5p_c': 14,
         'accoa_c': 13,
         'f6p_c': 12,
         'gly_c': 12,
         'akg_c': 11,
         'ctp_c': 10,
         'fmn_c': 10,
         '2dr1p_c': 10,
         'dhap_c': 9,
         'ser__L_c': 9,
         'gtp_c': 8,
         'cit_c': 8,
         'ala__L_c': 8,
         'thmpp_c': 8,
         'asp__L_c': 8,
         'uacgam_c': 8,
         '2

In [66]:
# all metabolites from unbalanced reactions
# this list is very important later on for some of the functions
unbalanced_mets = list(metabolite_counter_compartment.keys())

## scrape BIGG database - reactions
only needs to be executed once to get the csv file with the info, afterwards just read the csv file

In [14]:
# Get list of all universal reactions
base_url = "http://bigg.ucsd.edu/api/v2/"
list_url = base_url + "universal/reactions"
response = requests.get(list_url)

# check if request is going through
if response.status_code != 200:
    raise Exception("Failed to fetch reaction list")

reactions = response.json()["results"]
print(f"Found {len(reactions)} reactions. Fetching details...")

# Function that fetches specific information for one reaction
def fetch_reaction_details(reaction):
    bigg_id = reaction.get("bigg_id", "")
    name = reaction.get("name", "")
    url = f"{base_url}universal/reactions/{bigg_id}"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            metabolites = data.get("metabolites", [])

            # Safe quoting
            safe_name = str(name).replace('"', "'")

            return {
                "bigg_id": bigg_id,
                "name": f'"{safe_name}"',
                "metabolites": str(metabolites)
            }
    except Exception as e:
        print(f"Error with {bigg_id}: {e}")

    # Fallback if request fails
    safe_name = str(name).replace('"', "'")
    return {
        "bigg_id": bigg_id,
        "name": f'"{safe_name}"',
        "metabolites": "[]"
    }

# Use ThreadPoolExecutor to parallelize requests
results = []
with ThreadPoolExecutor(max_workers=35) as executor:
    futures = [executor.submit(fetch_reaction_details, rxn) for rxn in reactions]
    for i, future in enumerate(as_completed(futures)):
        results.append(future.result())
        if i % 500 == 0:
            print(f"{i}/{len(reactions)} done...")

Found 28302 reactions. Fetching details...
0/28302 done...
500/28302 done...
1000/28302 done...
1500/28302 done...
2000/28302 done...
2500/28302 done...
3000/28302 done...
3500/28302 done...
4000/28302 done...
4500/28302 done...
5000/28302 done...
5500/28302 done...
6000/28302 done...
6500/28302 done...
7000/28302 done...
7500/28302 done...
8000/28302 done...
8500/28302 done...
9000/28302 done...
9500/28302 done...
10000/28302 done...
10500/28302 done...
11000/28302 done...
11500/28302 done...
12000/28302 done...
12500/28302 done...
13000/28302 done...
13500/28302 done...
14000/28302 done...
14500/28302 done...
15000/28302 done...
15500/28302 done...
16000/28302 done...
16500/28302 done...
17000/28302 done...
17500/28302 done...
18000/28302 done...
18500/28302 done...
19000/28302 done...
19500/28302 done...
20000/28302 done...
20500/28302 done...
21000/28302 done...
21500/28302 done...
22000/28302 done...
22500/28302 done...
23000/28302 done...
23500/28302 done...
24000/28302 done...
2

In [ ]:
df_bigg_rea = pd.DataFrame(results)

# add a equation column to the bigg reaction df that includes the reaction equation in a format that we can directly use to overwrite the model equation with
df_bigg_rea["equation"] = None

for i in range(0, len(df_bigg_rea)):
    dict_equation = {}
    for met in df_bigg_rea["metabolites"][i]:
        name = met["bigg_id"] + "_" + met["compartment_bigg_id"]
        dict_equation[name] = met["stoichiometry"]

    df_bigg_rea.at[i, 'equation'] = dict_equation

# Save to CSV
# df_bigg_rea.to_csv("../bigg_reactions_complete.csv", index=False, quoting=csv.QUOTE_MINIMAL)

In [67]:
# Read CSV
df_bigg_rea = pd.read_csv("../Datasets/BIGG/bigg_reactions_complete.csv", quotechar='"', usecols=['bigg_id', 'name', 'metabolites', 'equation'])
df_bigg_rea["metabolites"] = df_bigg_rea["metabolites"].apply(ast.literal_eval)
df_bigg_rea["equation"] = df_bigg_rea["equation"].apply(ast.literal_eval)

In [77]:
df_bigg_rea

,bigg_id,name,metabolites,equation
0,10FTHF7GLUtm,"""7-glutamyl-10FTHF transport, mitochondrial""","[{'bigg_id': '10fthf7glu', 'name': '10-formylt...","{'10fthf7glu_c': 1.0, '10fthf7glu_m': -1.0}"
1,11DOCRTSLtm,"""11-deoxycortisol intracellular transport""","[{'bigg_id': '11docrtsl', 'name': '11docrtsl c...","{'11docrtsl_c': -1.0, '11docrtsl_m': 1.0}"
2,11DOCRTSTRNtr,"""11-deoxycorticosterone intracellular transport""","[{'bigg_id': '11docrtstrn', 'name': '11-Deoxyc...","{'11docrtstrn_c': -1.0, '11docrtstrn_r': 1.0}"
3,10FTHF5GLUtm,"""5-glutamyl-10FTHF transport, mitochondrial""","[{'bigg_id': '10fthf5glu', 'name': '10-formylt...","{'10fthf5glu_c': 1.0, '10fthf5glu_m': -1.0}"
4,10FTHF7GLUtl,"""7-glutamyl-10FTHF transport, lysosomal""","[{'bigg_id': '10fthf7glu', 'name': '10-formylt...","{'10fthf7glu_c': -1.0, '10fthf7glu_l': 1.0}"
...,...,...,...,...
28297,ZYMSTESTH_SC,"""Zymosterol ester hydrolase yeast specific""","[{'bigg_id': 'h', 'name': 'H+', 'compartment_b...","{'h_c': 1.0, 'h2o_c': -1.0, 'hdca_c': 0.02, 'h..."
28298,ZYMSTt,"""Zymosterol reversible transport""","[{'bigg_id': 'zymst', 'name': 'Zymosterol C27H...","{'zymst_c': 1.0, 'zymst_e': -1.0}"
28299,ZYMSTR,"""Zymosterol reductase""","[{'bigg_id': 'h', 'name': 'H+', 'compartment_b...","{'h_c': -1.0, 'nadp_c': 1.0, 'nadph_c': -1.0, ..."
28300,ZN2tpp,"""Zinc transport in via permease (no H+)""","[{'bigg_id': 'zn2', 'name': 'Zinc', 'compartme...","{'zn2_c': 1.0, 'zn2_p': -1.0}"


## scrape BIGG database - metabolites
only needs to be executed once to get the csv file with the info, afterwards just read the csv file

In [1]:
# this downloads metabolite information for all BIGG metabolites with their bigg ID, name, formulae and charge and saves it to a csv file
# there are 9088 metabolites


# Get list of all universal metabolites
base_url = "http://bigg.ucsd.edu/api/v2/"
list_url = base_url + "universal/metabolites"
response = requests.get(list_url)

# check if request is going through
if response.status_code != 200:
    raise Exception("Failed to fetch metabolite list")

metabolites = response.json()["results"]
print(f"Found {len(metabolites)} metabolites. Fetching details...")

# function that fetches specific information for one metabolite, i.e. BIGG ID, name, formulas and charges
def fetch_metabolite_details(met):
    bigg_id = met.get("bigg_id", "")
    name = met.get("name", "")
    url = f"{base_url}universal/metabolites/{bigg_id}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()  # converts JSON response to a dictionary (data)
            formulae = data.get("formulae", [])  # if formula not available, use empty list []
            charges = data.get("charges", [])

            # Safe quoting for CSV
            safe_name = str(name).replace('"', "'")
            name = f'"{safe_name}"'

            return {
                "bigg_id": bigg_id,
                "name": name,
                "formulas": str(formulae),
                "charges": str(charges)
            }
    except Exception as e:
        print(f"Error with {bigg_id}: {e}")

    safe_name = str(name).replace('"', "'")
    name = f'"{safe_name}"'
    return {
        "bigg_id": bigg_id,
        "name": name,
        "formulas": "[]",
        "charges": "[]"
    }

# Use ThreadPoolExecutor to parallelise requests
results = []
with ThreadPoolExecutor(max_workers=25) as executor:
    futures = [executor.submit(fetch_metabolite_details, met) for met in metabolites]
    for i, future in enumerate(as_completed(futures)):
        results.append(future.result())
        if i % 500 == 0:
            print(f"{i}/{len(metabolites)} done...")

# Save to CSV
df = pd.DataFrame(results)
df.to_csv("../bigg_metabolites_complete.csv", index=False, quoting=csv.QUOTE_MINIMAL)
print("Saved to bigg_metabolites_complete.csv")

Found 9088 metabolites. Fetching details...
0/9088 done...
500/9088 done...
1000/9088 done...
1500/9088 done...
2000/9088 done...
2500/9088 done...
3000/9088 done...
3500/9088 done...
4000/9088 done...
4500/9088 done...
5000/9088 done...
5500/9088 done...
6000/9088 done...
6500/9088 done...
7000/9088 done...
7500/9088 done...
8000/9088 done...
8500/9088 done...
9000/9088 done...
Saved to bigg_metabolites_complete.csv


In [68]:
# Read previously created CSV with all BIGG metabolites
df_bigg_met = pd.read_csv("../Datasets/BIGG/bigg_metabolites_complete.csv", quotechar='"')

# Convert stringified lists back to real lists
df_bigg_met["formulas"] = df_bigg_met["formulas"].apply(ast.literal_eval)
df_bigg_met["charges"] = df_bigg_met["charges"].apply(ast.literal_eval)


In [22]:
df_bigg_met

,bigg_id,name,formulas,charges
0,10fthf6glu,"""10-formyltetrahydrofolate-[Glu](6)""",[C45H51N12O22],[-7]
1,10fthf,"""10-Formyltetrahydrofolate""",[C20H21N7O7],[-2]
2,10fthfglu__L,"""10-Formyltetrahydrofolyl L-glutamate""",[C25H28N8O10],[]
3,10fthf5glu,"""10-formyltetrahydrofolate-[Glu](5)""",[C40H45N11O19],[-6]
4,10m3ouACP,"""10-methyl-3-oxo-undecanoyl-ACP""",[C23H41N2O9PRS],[0]
...,...,...,...,...
9083,zymstest_SC,"""Zymosterol ester yeast specific C1694H2993O101""",[C1694H2993O101],[0]
9084,xylu__L,"""L-Xylulose""",[C5H10O5],[0]
9085,zymst,"""Zymosterol C27H44O""",[C27H44O],[0]
9086,zymstnl,"""5alpha-cholest-8-en-3beta-ol""",[C27H46O],[0]


## scrape BIGG database - model

In [ ]:
# We again need to scrape bigg for the information in what model a reaction occurs
# this code needs 8min to execute but if we have the resulting csv file we dont need to execute this cell anymore
base_url = "http://bigg.ucsd.edu/api/v2/universal/reactions"
response = requests.get(base_url)

# check if request is going through
if response.status_code != 200:
    raise Exception("Failed to fetch metabolite list")

reactions = response.json()["results"]

def fetch_reaction_details(rxn):
    bigg_id = rxn["bigg_id"]
    url = f"{base_url}/{bigg_id}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            bigg_model = data.get("models_containing_reaction", [])

            bigg_model_id = [model["bigg_id"] for model in bigg_model]

            return {
                "bigg_id": bigg_id,
                "name": rxn["name"],
                "bigg_models": list(set(bigg_model_id))
            }

    except Exception as e:
        print(f"Error with {bigg_id}: {e}")
    return {
        "bigg_id": bigg_id,
    }

# Use ThreadPoolExecutor to parallelise requests
results = []
with ThreadPoolExecutor(max_workers=25) as executor:
    futures = [executor.submit(fetch_reaction_details, rxn) for rxn in reactions]
    for i, future in enumerate(as_completed(futures)):
        results.append(future.result())

rxn_model_origin = pd.DataFrame(results)
# rxn_model_origin.to_csv("../reactions_bigg_model_origin.csv", index=False, quoting=csv.QUOTE_MINIMAL)

In [ ]:
rxn_model_origin = pd.read_csv("../reactions_bigg_model_origin.csv", quotechar='"')
rxn_model_origin["bigg_models"] = rxn_model_origin["bigg_models"].apply(ast.literal_eval)

In [ ]:
# get models that have reactions that ONLY occur in this model
filtered_origins = set(([row[0] for row in rxn_model_origin["bigg_models"] if len(row) == 1]))

In [ ]:
filtered_origins

In [ ]:
# Filter reactions that occur in only one model
rxns_single_origin = rxn_model_origin[rxn_model_origin["bigg_models"].apply(lambda x: len(x) == 1)].copy()

# Extract the model each of those reactions comes from
rxns_single_origin["only_model"] = rxns_single_origin["bigg_models"].apply(lambda x: x[0])

# Count how many times each model appears
model_origin_counts = rxns_single_origin["only_model"].value_counts()

print(model_origin_counts)

In [ ]:
# To give carveme soft constraints we need a tsv file with the reaction IDs and in the second columns a 0, i.e. we want to exclude these reactions
# https://carveme.readthedocs.io/en/latest/advanced.html#experimental-constraints
false_models = ["iYO844"]  # list of models that we found are "problematic"

# Step 1: Filter reactions where *all* associated models are in false_models
# i.e. For the given list "models" (which is one row of the rxn_model_origin df), return True only if every element m (every model from that row) is in the false_models list.
false_rxns = rxn_model_origin[
    rxn_model_origin["bigg_models"].apply(lambda models: all(m in false_models for m in models))
]

# Step 2: Extract BiGG reaction IDs
false_rxn_ids = false_rxns["bigg_id"].tolist()

# Step 3: Create DataFrame with ID and 0
output_df = pd.DataFrame({
    "ID": [f'R_{rxn_id}' for rxn_id in false_rxn_ids],  # Prefix "R_" because this is how reactions are named in the universal model that carveme uses
    "value": [0] * len(false_rxn_ids) # 0 because then CarveMe excludes those reactions
})


In [ ]:
output_df

In [ ]:
output_df.to_csv("../filtered_false_reactions.tsv", sep="\t", index=False, header=False)

## Extract Metabolite info out of models
we need infos about metabolites from out models so we can compare it to the BIGG info that we just downloaded

In [69]:
# get metabolite info for all 7 models (i.e. formula and charge state) and save the 7 df's in a dict
model_mets = {f"AA{i}_mets": extract_met_info_model(models[f"AA{i}"]) for i in range(1, 8)}

In [70]:
model_mets["AA1_mets"]

,bigg_id,model_id,model_formula,model_charge
0,10fthf,10fthf_c,C20H21N7O7,-2
1,12dgr120,12dgr120_c,C27H52O5,0
2,12dgr120,12dgr120_p,C27H52O5,0
3,12dgr140,12dgr140_c,C31H60O5,0
4,12dgr140,12dgr140_p,C31H60O5,0
...,...,...,...,...
1536,xylb,xylb_e,C10H18O9,0
1537,xylu__D,xylu__D_c,C5H10O5,0
1538,zn2,zn2_c,Zn,2
1539,zn2,zn2_e,Zn,2


In [71]:
# merge the metabolite info from the models with the bigg info; creates 2 columns to show if charge/formula info match between model and bigg
# saves all 7 df's in a dict model_merged but for easier access to the individual df's, there are saved as objects (e.g. AA1_merged) but these are still linked to the dict
model_merged = {f"AA{i}_merged": compare_bigg_modelMets(model_mets[f"AA{i}_mets"], unbalanced_mets) for i in range(1, 8)}
AA1_merged, AA2_merged, AA3_merged, AA4_merged, AA5_merged, AA6_merged, AA7_merged = [model_merged[f"AA{i}_merged"] for i in range(1, 8)]

In [72]:
AA1_merged

,bigg_id,model_id,model_formula,model_charge,name,formulas,charges,charge_match,formula_match,unbalanced
0,10fthf,10fthf_c,C20H21N7O7,-2,"""10-Formyltetrahydrofolate""",[C20H21N7O7],[-2],True,True,True
1,12dgr120,12dgr120_c,C27H52O5,0,"""1,2-Diacyl-sn-glycerol (didodecanoyl, n-C12:0)""",[C27H52O5],[0],True,True,True
2,12dgr120,12dgr120_p,C27H52O5,0,"""1,2-Diacyl-sn-glycerol (didodecanoyl, n-C12:0)""",[C27H52O5],[0],True,True,True
3,12dgr140,12dgr140_c,C31H60O5,0,"""1,2-Diacyl-sn-glycerol (ditetradecanoyl, n-C1...",[C31H60O5],[0],True,True,True
4,12dgr140,12dgr140_p,C31H60O5,0,"""1,2-Diacyl-sn-glycerol (ditetradecanoyl, n-C1...",[C31H60O5],[0],True,True,True
...,...,...,...,...,...,...,...,...,...,...
1536,xylb,xylb_e,C10H18O9,0,"""Xylobiose""",[C10H18O9],[],False,True,False
1537,xylu__D,xylu__D_c,C5H10O5,0,"""D-Xylulose""",[C5H10O5],[0],True,True,False
1538,zn2,zn2_c,Zn,2,"""Zinc""",[Zn],[2],True,True,False
1539,zn2,zn2_e,Zn,2,"""Zinc""",[Zn],[2],True,True,False


In [73]:
# extracts just the rows where charge and/or formula dont match with bigg info and saves them into a dict with the seven df's
model_mismatch = {f"AA{i}_mismatch": get_mismatches_after_merge(model_merged[f"AA{i}_merged"]) for i in range(1, 8)}

In [74]:
model_mismatch["AA1_mismatch"]

,model_id,bigg_id,model_charge,charges,model_formula,formulas,charge_match,formula_match,unbalanced
24,1btol_c,1btol,0,[],C4H10O,[C4H10O],False,True,False
27,1p2cbxl_c,1p2cbxl,0,[-1],C5H6NO2,[C5H6NO2],False,True,True
30,23ddhb_c,23ddhb,0,[-1],C7H7O4,[C7H7O4],False,True,True
44,2agpe160_c,2agpe160,0,[0],C21H44NO7P,[C21H44NO7P1],True,False,True
45,2agpe160_p,2agpe160,0,[0],C21H44NO7P,[C21H44NO7P1],True,False,False
...,...,...,...,...,...,...,...,...,...
1519,val__D_p,val__D,0,[0],C5H9NO2,[C5H11NO2],True,False,False
1531,xyl3_c,xyl3,0,[],C15H26O13,[C15H26O13],False,True,False
1532,xyl3_e,xyl3,0,[],C15H26O13,[C15H26O13],False,True,False
1535,xylb_c,xylb,0,[],C10H18O9,[C10H18O9],False,True,False


In [75]:
# confusion matrix to show for how many metabolites there are differences in the infos between current model and bigg
get_confmat_charge_formula(AA1_merged)

   charge_match  formula_match  count
0         False          False    181
1         False           True    191
2          True          False    102
3          True           True   1067


In [76]:
# function also takes dict with all the 7 df's and creates one big confusion matrix
get_confmat_charge_formula(model_merged)

  charge_match forumla_match   AA1   AA2   AA3   AA4   AA5   AA6   AA7
0        False         False   181   205   157   126   136   194   191
1        False          True   191   222   238   182   191   210   233
2         True         False   102    76   103   111    98    94    73
3         True          True  1067  1186  1040  1331  1057  1177  1171


In [77]:
# for all metabolites in AA1, there are 816 metabolites (53%) that are not part of unbalanced reactions and 716 metabolites that are in unbalanced reactions
# metabolites are only part
AA1_merged['unbalanced'].value_counts()

unbalanced
False    828
True     713
Name: count, dtype: int64

In [78]:
# if we only look at metabolites where BIGG infos and model infos do NOT match, we now have 319 metabolites (66.6%) that are in unbalanced reactions and only 33.4% of these metabolites are in balanced reactions
model_mismatch["AA1_mismatch"]['unbalanced'].value_counts()

unbalanced
True     313
False    161
Name: count, dtype: int64

In [79]:
combo_counts = AA1_merged.groupby(['unbalanced', 'charge_match', 'formula_match']).size().reset_index(name='count')
print(combo_counts)
# "False True True" is optimal case, i.e. metabolite has infos that matches with bigg and is not in any unbalanced reaction

   unbalanced  charge_match  formula_match  count
0       False         False          False     62
1       False         False           True     54
2       False          True          False     45
3       False          True           True    667
4        True         False          False    119
5        True         False           True    137
6        True          True          False     57
7        True          True           True    400


In [80]:
AA6_merged['unbalanced'].value_counts()

unbalanced
False    932
True     743
Name: count, dtype: int64

In [81]:
model_mismatch["AA6_mismatch"]['unbalanced'].value_counts()

unbalanced
True     326
False    172
Name: count, dtype: int64

In [82]:
combo_counts = AA6_merged.groupby(['unbalanced', 'charge_match', 'formula_match']).size().reset_index(name='count')
print(combo_counts)

   unbalanced  charge_match  formula_match  count
0       False         False          False     75
1       False         False           True     52
2       False          True          False     45
3       False          True           True    760
4        True         False          False    119
5        True         False           True    158
6        True          True          False     49
7        True          True           True    417


## Overwrite model with BIGG information
We want to try if the information on BIGG is valuable to our model to help with the big amount of charge unbalanced reactions.
That means for reactions or rather metabolites were the bigg information is different to our model info, we can try to overwrite it with the bigg info.

I am only overwriting model info with BIGG info if the BIGG info only gives one charge/formula. If they're multiple possible charge states/formulas I should do manual curation. I also only overwrite it when the metabolite is part of an unbalanced reaction.

### Create a copies of our models
with these I can introduces changes and then can compare it to the OG model

In [83]:
# read the xml files a second time to create "copies" of our models that we can try to curate and compare to the original model

model_path = "../Models/02_afterCuration/"

models_curation = {}
for model_name in (f for f in os.listdir(model_path) if f.endswith(".xml")):
    model = read_sbml_model(f"{model_path}{model_name}")
    model.solver = "cplex"
    name = str(model_name[:3]+"_curate")
    models_curation[name] = model

models_curation = {key: models_curation[key] for key in sorted(models_curation.keys())}  # sorts the dictionary alphabetically
AA1_curate, AA2_curate, AA3_curate, AA4_curate, AA5_curate, AA6_curate, AA7_curate = [models_curation[f"AA{i}_curate"] for i in range(1, 8)]

### Use BIGG metabolite info to overwrite model metabolite infos
--> only overwrite when bigg info is unambiguously

In [84]:
# Overwrite model metabolite info with BIGG info if the BIGG info is unambiguously, i.e. only one charge state/formula
def overwrite_with_BIGG_metabolites(model, merged_df):
    n_unbalanced = len(check_balance(model, print_results=False))
    for i in range(0,len(merged_df)):
        # check if there is only one charge state
        if len(merged_df["charges"][i]) == 1:
            model.metabolites.get_by_id(merged_df["model_id"][i]).charge = int(merged_df["charges"][i][0])

        else:  # we only need this to get the right datatype (int) for the charge state to save the model later on because apparently it got fucked up
            model.metabolites.get_by_id(merged_df["model_id"][i]).charge = int(model.metabolites.get_by_id(merged_df["model_id"][i]).charge)

        # check if there is only one formula (and at least one charge state because otherwise that model formula could be right)
        if len(merged_df["formulas"][i]) == 1 and len(merged_df["charges"][i]) != 0:
            if not "X" in merged_df["formulas"][i][0] and not "R" in merged_df["formulas"][i][0]:
                model.metabolites.get_by_id(merged_df["model_id"][i]).formula = merged_df["formulas"][i][0]

    n_unbalanced_update = len(check_balance(model, print_results=False))
    print(f'{model.id}: There were {n_unbalanced} unbalanced reactions before and now there are {n_unbalanced_update} after overwriting metabolite info with BIGG data.')

In [85]:
overwrite_with_BIGG_metabolites(AA1_curate, AA1_merged)
overwrite_with_BIGG_metabolites(AA2_curate, AA2_merged)
overwrite_with_BIGG_metabolites(AA3_curate, AA3_merged)
overwrite_with_BIGG_metabolites(AA4_curate, AA4_merged)
overwrite_with_BIGG_metabolites(AA5_curate, AA5_merged)
overwrite_with_BIGG_metabolites(AA6_curate, AA6_merged)
overwrite_with_BIGG_metabolites(AA7_curate, AA7_merged)

AA1: There were 448 unbalanced reactions before and now there are 198 after overwriting metabolite info with BIGG data.
AA2: There were 433 unbalanced reactions before and now there are 183 after overwriting metabolite info with BIGG data.
AA3: There were 398 unbalanced reactions before and now there are 180 after overwriting metabolite info with BIGG data.
AA4: There were 401 unbalanced reactions before and now there are 161 after overwriting metabolite info with BIGG data.
AA5: There were 384 unbalanced reactions before and now there are 190 after overwriting metabolite info with BIGG data.
AA6: There were 440 unbalanced reactions before and now there are 190 after overwriting metabolite info with BIGG data.
AA7: There were 454 unbalanced reactions before and now there are 175 after overwriting metabolite info with BIGG data.


### Use BIGG reactions to overwrite model reactions info

In [86]:
def overwrite_with_BIGG_reactions(model):
    unbalanced_rxns = check_balance(model, print_results=False)
    unbalanced_rxns = [r.id for r in unbalanced_rxns]

    for rxn in unbalanced_rxns:

        new_react = df_bigg_rea[df_bigg_rea['bigg_id'] == rxn]["equation"].iloc[0]

        new_mets_dict = {model.metabolites.get_by_id(met_id): coeff for met_id, coeff in new_react.items()}

        reaction = model.reactions.get_by_id(rxn)
        reaction.subtract_metabolites(reaction.metabolites)
        reaction.add_metabolites(new_mets_dict)

    unbalanced_rxns_after = check_balance(model, print_results=False)
    unbalanced_rxns_after = [r.id for r in unbalanced_rxns_after]

    print(f'{model.id}: There were {len(unbalanced_rxns)} unbalanced reactions before and now there are {len(unbalanced_rxns_after)} after overwriting reaction info with BIGG data.')

In [87]:
overwrite_with_BIGG_reactions(AA1_curate)
overwrite_with_BIGG_reactions(AA2_curate)
overwrite_with_BIGG_reactions(AA3_curate)
overwrite_with_BIGG_reactions(AA4_curate)
overwrite_with_BIGG_reactions(AA5_curate)
overwrite_with_BIGG_reactions(AA6_curate)
overwrite_with_BIGG_reactions(AA7_curate)

AA1: There were 198 unbalanced reactions before and now there are 194 after overwriting reaction info with BIGG data.
AA2: There were 183 unbalanced reactions before and now there are 179 after overwriting reaction info with BIGG data.
AA3: There were 180 unbalanced reactions before and now there are 175 after overwriting reaction info with BIGG data.
AA4: There were 161 unbalanced reactions before and now there are 157 after overwriting reaction info with BIGG data.
AA5: There were 190 unbalanced reactions before and now there are 184 after overwriting reaction info with BIGG data.
AA6: There were 190 unbalanced reactions before and now there are 186 after overwriting reaction info with BIGG data.
AA7: There were 175 unbalanced reactions before and now there are 171 after overwriting reaction info with BIGG data.


In [88]:
total_unique = set()
for model in models_curation.values():
    aa1_rxns = check_balance(model)
    total_unique.update(set([rxn.id for rxn in aa1_rxns.keys()]))
print(f"There are in total {len(total_unique)} unique reactions in the models")

There are 194 unbalanced reactions in AA1
There are 179 unbalanced reactions in AA2
There are 175 unbalanced reactions in AA3
There are 157 unbalanced reactions in AA4
There are 184 unbalanced reactions in AA5
There are 186 unbalanced reactions in AA6
There are 171 unbalanced reactions in AA7
There are in total 376 unique reactions in the models


In [89]:
check_balance(AA3_curate)

There are 175 unbalanced reactions in AA3


{<Reaction 23CTI1 at 0x7075e53434c0>: {'H': -1.0},
 <Reaction AACPS2 at 0x7075e6ff2b60>: {'charge': -2.0},
 <Reaction AACPS3 at 0x7075e54144f0>: {'charge': -1.0},
 <Reaction AACPS4 at 0x7075e4a3bd00>: {'charge': -2.0},
 <Reaction AACPS6 at 0x7075e5dcfb50>: {'charge': -2.0},
 <Reaction AACPS7 at 0x7075e711ce20>: {'charge': -1.0},
 <Reaction AACPS9 at 0x7075e7a56140>: {'charge': -2.0},
 <Reaction ACOAD1f at 0x7075e6507a90>: {'charge': -2.0},
 <Reaction ACOAD1fr at 0x7075e63a0d30>: {'charge': 2.0},
 <Reaction ACOAD21f at 0x7075e656ea10>: {'charge': -2.0},
 <Reaction ACOAD23f at 0x7075e656fca0>: {'charge': -2.0},
 <Reaction ACOAD24f at 0x7075e656d930>: {'charge': -2.0},
 <Reaction ACOAD26f at 0x7075e63a2410>: {'charge': -2.0},
 <Reaction ACOAD27f at 0x7075e63e0c70>: {'charge': -2.0},
 <Reaction ACOAD28f at 0x7075e63e0cd0>: {'charge': -2.0},
 <Reaction ACOAD2f at 0x7075e656fb50>: {'charge': -2.0},
 <Reaction ACOAD31f at 0x7075e64182e0>: {'charge': -2.0},
 <Reaction ACOAD32f at 0x7075e644e35

# Manual changes
These following changes were made in a manual curation process, i.e. checking all imbalanced reactions by hand and changing mostly charge and formula of metabolites according to databases like BIGG, SEED and MetaCyc.
Some metabolites were not overwritten by BIGG info in the previous step because BIGG has multiple charge states/formulas and now we need to decide what fits the model the best.
Priority is to get rid of mass imbalances and second is charge imbalances, because we definitely want a stoichiometry consistent model.

## Functions to overwrite metabolites and reactions

In [191]:
def overwrite_charge(model, met_id, new_charge):
    if met_id in model.metabolites:
        model.metabolites.get_by_id(met_id).charge = new_charge

In [192]:
def overwrite_formula(model, met_id, new_formula):
    if met_id in model.metabolites:
        model.metabolites.get_by_id(met_id).formula = new_formula

In [92]:
def overwrite_reaction(model, rxn_id, new_rxn_dict):
    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        rxn.subtract_metabolites(rxn.metabolites)
        rxn.add_metabolites(new_rxn_dict)

In [93]:
def delete_metabolite(model, met_id):
    if met_id in model.metabolites:

        if len(model.metabolites.get_by_id(met_id).reactions) == 0:
            met = model.metabolites.get_by_id(met_id)
            model.metabolites.remove(met)

        else:
            print(f'metabolite {met_id} cannot be deleted from {model.id} because of reaction(s): {model.metabolites.get_by_id(met_id).reactions}')


In [94]:
def delete_reaction(model, rxn_id):
    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        model.remove_reactions([rxn])

In [95]:
def delete_duplicate_reaction(model, rxn_old, rxn_new, gpr = False):
    if rxn_old in model.reactions and rxn_new in model.reactions:
        rxn_o = model.reactions.get_by_id(rxn_old)
        rxn_n = model.reactions.get_by_id(rxn_new)

        if gpr == True: # combine GPR aka keep one of them if both have something in common or are empty
            if len(rxn_n.gene_reaction_rule) == 0 and len(rxn_o.gene_reaction_rule) == 0:
                rxn_n.gene_reaction_rule = ''
            if len(rxn_n.gene_reaction_rule) == 0:
                rxn_n.gene_reaction_rule = rxn_o.gene_reaction_rule
            if len(rxn_o.gene_reaction_rule) == 0:
                rxn_n.gene_reaction_rule = rxn_n.gene_reaction_rule
            if rxn_n.gene_reaction_rule in rxn_o.gene_reaction_rule or rxn_o.gene_reaction_rule in rxn_n.gene_reaction_rule: # check if gpr's overlap
                rxn_n.gene_reaction_rule =  rxn_n.gene_reaction_rule if len(rxn_n.gene_reaction_rule) >= len(rxn_o.gene_reaction_rule) else rxn_o.gene_reaction_rule

            else:
                return # if gpr's dont overlap we return and dont delete the reaction

        model.remove_reactions([rxn_o])

## Overwrite

In [294]:
def overwrite_manual(model):
    # first all changes to metabolites, i.e. charges and formulas
    # afterwards changes for reactions, i.e. changing stoichiometry, replacing/deleting metabolites (especially H)
    # last deletions (mostly reactions if duplicate but also metabolites)
    # every category is alphabetically sorted

    # first: metabolites
    overwrite_formula(model, "2ameph_p", "C2H7NO3P") # og = C2H8NO3P, charge was changed from 0 to -1 automatically with bigg and formula now also needed to be changed
    overwrite_formula(model, "2ameph_e", "C2H7NO3P")
    overwrite_formula(model, "2ameph_c", "C2H7NO3P")
    overwrite_charge(model, "2dhphaccoa_c", -4) #og = 0; according to seed https://modelseed.org/biochem/compounds/cpd16740
    overwrite_charge(model, "2mpdhl_c", -1)
    overwrite_charge(model, "23dhbzs3_c", -1) # og = 0, -1 alterative in bigg

    overwrite_formula(model, "23dhbzs3_e", "C30H29N3O16") #one more H to fit charge
    overwrite_charge(model, "3hbutACP_c", -1) # og = 0, -1 alterative in bigg
    overwrite_formula(model, "3hsa_c", "C19H24O3")
    overwrite_charge(model, "3sala_c", -1) # og = 0
    overwrite_formula(model, "3sala_c", "C3H6NO4S")
    overwrite_formula(model, "34dhsa_c", "C19H24O4")
    overwrite_charge(model, "4cml_c", -2)
    overwrite_formula(model, "4cml_c", "C7H4O6")
    overwrite_charge(model, "4hoxpac_c", -1) # og = 0, -1 alterative in bigg
    overwrite_charge(model, "4hoxpac_e", -1)
    overwrite_charge(model, "4hoxpac_p", -1)
    overwrite_formula(model, "49dsha_c", "C19H23O6")
    overwrite_charge(model, "49dsha_c", -1)
    overwrite_charge(model, "5aizc_c", -3)
    overwrite_formula(model, "5mthf_c", "C20H24N7O6") # C20H24N7O6, https://www.genome.jp/entry/C00440
    overwrite_formula(model, "5ohhipcoa_c", "C34H50N7O19P3S")
    overwrite_charge(model, "5ohhipcoa_c", -4)
    overwrite_charge(model, "5ohhip_c", -2)
    overwrite_charge(model, "6pgg_c", -2) # og = 0, -2 alterative in bigg and in accordance with ecoli
    overwrite_formula(model, "9ohadd_c", "C19H24O3")

    overwrite_charge(model, "aad_c", -2)
    overwrite_formula(model, "abg4_c", "C12H12N2O5") # C12H11N2O5; https://biocyc.org/compound?orgid=META&id=CPD0-889
    overwrite_charge(model, "abg4_c", -2) # og = 0
    overwrite_formula(model, "abg4_e", "C12H12N2O5")
    overwrite_charge(model, "abg4_e", -2)
    overwrite_formula(model, "acadl_c", "C12H14N5O8P") # og = C12H14N5O8P; https://pubchem.ncbi.nlm.nih.gov/compound/440867 with h16 charge = 0, we have -1 charge
    overwrite_charge(model, "acadl_c", -2) # https://pubchem.ncbi.nlm.nih.gov/compound/440867 with h16 is charge = 0 and we have H14, so we need -2
    overwrite_charge(model, "ACP_c", 0)
    overwrite_charge(model, "actACP_c", -1)
    overwrite_charge(model, "acysbmn_e", -1) # og = 0, -1 according to metacyc with same formula https://metacyc.org/compound?orgid=META&id=CPD1G-185
    overwrite_charge(model, "acysbmn_c", -1) # og = 0, -1
    overwrite_charge(model, "ah6p__D_c", -2) # og = 0, -2 must be because f6p is also -2 and they can directly converted into each other
    overwrite_charge(model, "air_c", -2)
    overwrite_charge(model, "amacald_c", 1)
    overwrite_formula(model, "amacald_c", "C2H6NO")
    overwrite_formula(model, "andrs14dn317dn_c", "C19H24O2")
    overwrite_formula(model, "apoACP_c", "C373H582N94O136S2") # og = C373H583N94O136S2; charge was changed from 1 to 0 and now the amount of H also reflects that
    overwrite_charge(model, "aso3_c", -1)
    overwrite_charge(model, "aso3_e", -1)
    overwrite_charge(model, "aso3_p", -1)
    overwrite_formula(model, "aso3_c", "H2O3As")
    overwrite_formula(model, "aso3_e", "H2O3As")
    overwrite_formula(model, "aso3_p", "H2O3As")
    overwrite_formula(model, "aso4_c", "HO4As")
    overwrite_formula(model, "aso4_e", "HO4As")
    overwrite_formula(model, "aso4_p", "HO4As")

    overwrite_charge(model, "bmn_c", 2) # og = 0, to balance BMNMSHS (bmn is just imported for this reaction)
    overwrite_charge(model, "bmn_e", 2)
    overwrite_charge(model, "but2eACP_c", -1)

    overwrite_charge(model, "CCbuttc_c", -3) # og = -3; to balance reaction with 4cml_c
    overwrite_formula(model, "CCbuttc_c", "C7H3O6") # C7H3O6; to reflect charge change
    overwrite_formula(model, "cchol_c", "C27H42O3")
    overwrite_charge(model, "cdigmp_c", -2)
    overwrite_formula(model, "cholc3coa_c", "C43H66N7O18P3S")
    overwrite_formula(model, "cholc5coa_c", "C45H70N7O18P3S")
    overwrite_formula(model, "cholc8coa_c", "C48H76N7O18P3S")
    overwrite_formula(model, "cholenec3coa_c", "C43H64N7O18P3S")
    overwrite_formula(model, "cholenec5coa_c", "C45H68N7O18P3S")
    overwrite_formula(model, "cholenec8coa_c", "C48H74N7O18P3S")

    overwrite_formula(model, "decoa_c", "C31H48N7O17P3S")
    overwrite_charge(model, "dgal6p_c", -2)
    overwrite_charge(model, "dmlgnc_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd15951
    overwrite_charge(model, "dtbt_c", -1)

    overwrite_charge(model, "fad_c", -2)
    overwrite_charge(model, "fad_e", -2)
    overwrite_charge(model, "fad_p", -2)
    overwrite_charge(model, "fe3dhbzs3_c", 3) # og=0, alternative in bigg
    overwrite_formula(model, "fe3dhbzs3_c", "C30FeH29N3O16") # og = C30FeH28N3O16, with H29 is in bigg and ecoli
    overwrite_formula(model, "fe3dhbzs3_e", "C30FeH29N3O16")
    overwrite_charge(model, "fe3dhbzs3_e", 3) # 0 -> 3
    overwrite_formula(model, "fe3dhbzs3_p", "C30FeH29N3O16")
    overwrite_formula(model, "feoxam_c", "C25H46FeN6O8") # formula change according to bigg and ecoli
    overwrite_formula(model, "feoxam_e", "C25H46FeN6O8")
    overwrite_formula(model, "feoxam_p", "C25H46FeN6O8")
    overwrite_charge(model, "ficytc_c", 1)
    overwrite_charge(model, "fmcbtt_c", 2) # og = 0 but it has fe2 in it
    overwrite_charge(model, "fmn_c", -2)
    overwrite_formula(model, "fmn_c", "C17H19N4O9P")
    overwrite_charge(model, "fmn_e", -2)
    overwrite_formula(model, "fmn_e", "C17H19N4O9P")
    overwrite_charge(model, "fmn_p", -2)
    overwrite_formula(model, "fmn_p", "C17H19N4O9P")
    overwrite_charge(model, "focytc_c", 1)
    overwrite_charge(model, "fpram_c", -1)
    overwrite_formula(model, "fpram_c", "C8H15N3O8P")

    overwrite_charge(model, "g3p_c", -2)
    overwrite_charge(model, "g6p_A_c", -2)
    overwrite_formula(model, "galam6p_c", "C6H13NO8P") # https://biocyc.org/compound?orgid=META&id=D-GALACTOSAMINE-6-PHOSPHATE
    overwrite_charge(model, "galam6p_c", -1)
    overwrite_charge(model, "galam_p", +1) #og = 0 https://metacyc.org/compound?orgid=META&id=GALACTOSAMINE
    overwrite_formula(model, "galam_p", "C6H14NO5")
    overwrite_charge(model, "galam_e", +1)
    overwrite_formula(model, "galam_e", "C6H14NO5")
    overwrite_charge(model, "gcvHL_ADPr_c", -1)
    overwrite_formula(model, "gcvHL_ADPr_c", "C23H36N6O21P4S2")
    overwrite_charge(model, "gcvHL_nhLA_c", 0)
    overwrite_formula(model, "gcvHL_nhLA_c", "C8H16NO8P2S2")
    overwrite_charge(model, "gdptp_c", -7)
    overwrite_charge(model, "glutrna_c", -3)
    overwrite_formula(model, "glycogen_c", "C6H10O5")
    overwrite_charge(model, "gly_pro__L_c", 1)
    overwrite_formula(model, "gly_pro__L_c", "C7H13N2O3")
    overwrite_charge(model, "gly_pro__L_e", 1)
    overwrite_formula(model, "gly_pro__L_e", "C7H13N2O3")
    overwrite_formula(model, "gly_tyr_c", "C11H14N2O4")
    overwrite_formula(model, "gly_phe_c", "C11H14N2O3")
    overwrite_formula(model, "gly_leu_c", "C8H16N2O3")
    overwrite_formula(model, "gly_cys_c", "C5H10N2O3S")

    overwrite_formula(model, "hchol_c", "C27H44O2")
    overwrite_formula(model, "hcholc8coa_c", "C48H76N7O19P3S")
    overwrite_formula(model, "hcholc5coa_c", "C45H70N7O19P3S")
    overwrite_formula(model, "hcholc3coa_c", "C43H66N7O19P3S")
    overwrite_charge(model, "hdeACP_c", 0)
    overwrite_charge(model, "hethmpp_c", -2)
    overwrite_charge(model, "hemeO_c", -2)
    overwrite_formula(model, "hia_c", "C11H16O4")
    overwrite_formula(model, "hia_e", "C11H16O4")
    overwrite_formula(model, "hip_c", "C13H17O4")
    overwrite_charge(model, "hip_c", -1)
    overwrite_formula(model, "hipcoa_c", "C34H48N7O19P3S")
    overwrite_charge(model, "hipcoa_c", -4)
    overwrite_formula(model, "hipecoa_c", "C34H48N7O19P3S")
    overwrite_charge(model, "hipecoa_c", -4)
    overwrite_formula(model, "hipohcoa_c", "C34H50N7O20P3S")
    overwrite_charge(model, "hipohcoa_c", -4)
    overwrite_formula(model, "hipocoa_c", "C34H48N7O20P3S")
    overwrite_charge(model, "hipocoa_c", -4)
    overwrite_charge(model, "hmbpp_c", -4) # pubchem C5H12O8P2 with charge 0; model=C5H8O8P2, so charge must be -4

    overwrite_charge(model, "istfrnA_e", -2)
    overwrite_formula(model, "istfrnA_e", "C17FeH19N2O14")
    overwrite_charge(model, "istfrnB_e", +1)
    overwrite_formula(model, "istfrnB_e", "C16FeH22N2O11")

    overwrite_charge(model, "lysglugly_c", 0)
    overwrite_charge(model, "lysglugly_e", 0)

    overwrite_charge(model, "man6pglyc_c", -3) # og = 0; alternative in bigg and in accordance with ecoli
    overwrite_charge(model, "mbhn_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd15971
    overwrite_formula(model, "mcbtt_c", "C47H77N5O10") # was wrongly overwritten by a false bigg formula = [C43H71N5O10], metacyc also has the original one that was in the model
    overwrite_charge(model, "mcbtt_c", 0)
    overwrite_charge(model, "met_L_ala__L_c", -1)
    overwrite_charge(model, "met_L_ala__L_e", -1)
    overwrite_formula(model, "met_L_ala__L_c", "C8H15N2O3S")
    overwrite_formula(model, "met_L_ala__L_e", "C8H15N2O3S")
    overwrite_charge(model, "mhpglu_c", -4)
    overwrite_charge(model, "mi3p__D_c", -2) # og = 0, -2 according to bigg

    overwrite_formula(model, "Nforglu_c", "C6H7NO5")

    overwrite_charge(model, "ocACP_c", 0) # og = -1, 0 alterative in bigg and is in accordance with charge = 0 of ACP
    overwrite_formula(model, "ochol_c", "C27H42O2")
    overwrite_formula(model, "ocholc8coa_c", "C48H74N7O19P3S")
    overwrite_formula(model, "ocholc5coa_c", "C45H68N7O19P3S")
    overwrite_charge(model, "ocdcaACP_c", 0) # og = -1, 0 alterative in bigg and is in accordance with charge = 0 of ACP


    overwrite_charge(model, "pdima_c", -1) # 0
    overwrite_charge(model, "pdima_e", -1) # 0
    overwrite_charge(model, "peng_p", -1) # og = 0, https://modelseed.org/biochem/compounds/cpd03292
    overwrite_formula(model, "peng_p", "C16H17N2O4S") # og = C16H18N2O4S,https://modelseed.org/biochem/compounds/cpd03292
    overwrite_charge(model, "peng_e", -1)
    overwrite_formula(model, "peng_e", "C16H17N2O4S")
    overwrite_charge(model, "phdcacoa_c", -4) # og=0 but its coa
    overwrite_charge(model, "phdca_c", -1) # og = 0 but https://modelseed.org/biochem/compounds/cpd16013
    overwrite_charge(model, "phdca_e", -1)
    overwrite_formula(model, "phthdl_c", "C31H62O3") # C31H61O3
    overwrite_charge(model, "phthdl_c", 0) # 0
    overwrite_formula(model, "phthiocerol_c", "C32H66O3") # C32H65O3
    overwrite_charge(model, "phthiocerol_c", 0) # 0
    overwrite_charge(model, "ppad_c", -2)
    overwrite_charge(model, "ptd1ino160_c", -1)
    overwrite_charge(model, "pqqh2_c", -3)
    overwrite_charge(model, "pqqh2_p", -3)
    overwrite_charge(model, "ppgpp_c", -6)
    overwrite_formula(model, "prephth_c", "C32H61O5") # C32H60O5
    overwrite_charge(model, "prephth_c", -1) # 0
    overwrite_formula(model, "prephthACP_c", "C416H663N96O146PS3") # C416H662N96O146PS3
    overwrite_charge(model, "prephthACP_c", 0) # 0
    overwrite_charge(model, "prepphth_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd16028
    overwrite_charge(model, "prohisglu_c", -1) # og = -2; tripeptid pro-his-glu, only glu has -1 charge and other two are neutral
    overwrite_charge(model, "prohisglu_e", -1)

    overwrite_formula(model, "ribflv_c", "C17H20N4O6")
    overwrite_formula(model, "ribflv_e", "C17H20N4O6")

    # Salmochelin fixes (there are first fixes by Frowin in the apply mass balance function notebook)
    overwrite_formula(model, "salchsx_c", "C16H20NO11") # og C16H21NO11; https://pubchem.ncbi.nlm.nih.gov/compound/135397946
    overwrite_formula(model, "salchsx_e", "C16H20NO11")
    overwrite_formula(model, "salchsx_p", "C16H20NO11")
    overwrite_charge(model, "salchs2fe_c", 3) # to match salchs4fe
    overwrite_charge(model, "salchs2fe_p", 3)
    overwrite_charge(model, "salchs2fe_e", 3)
    #----- more S
    overwrite_charge(model, "salc_e", -1) # og = 0, -1 alterative in bigg
    overwrite_charge(model, "salc_c", -1)
    overwrite_charge(model, "scl_c", -7) # og = 0, -7 according to bigg
    overwrite_charge(model, "scys__L_c", -1)
    overwrite_charge(model, "ssaltpp_c", -3) # og = 0; 0 is not in bigg only -3 or -2
    overwrite_charge(model, "stfrnA_e", -5)
    overwrite_formula(model, "stfrnA_e", "C17H19N2O14")
    overwrite_charge(model, "stfrnA_c", -5)
    overwrite_formula(model, "stfrnA_c", "C17H19N2O14")
    overwrite_charge(model, "stfrnB_e", -2)
    overwrite_formula(model, "stfrnB_e", "C16H22N2O11")
    overwrite_charge(model, "stfrnB_c", -2)
    overwrite_formula(model, "stfrnB_c", "C16H22N2O11")

    overwrite_charge(model, "tag6p__D_c", -2)
    overwrite_charge(model, "tagdp__D_c", -4)
    overwrite_formula(model, "tagur_e", "C6H9O7") # make it the same as tagur_c
    overwrite_charge(model, "tamocta_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd16038
    overwrite_charge(model, "tdeACP_c", 0)
    overwrite_charge(model, "tmhexc_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd16050

    overwrite_charge(model, "udpacgal_c", -2) # og = 0, -2 alterative in bigg
    overwrite_charge(model, "udpacgal_p", -2) # og = 0; -2 alternative in bigg and in accordance with ecoli
    overwrite_charge(model, "udpacgal_e", -2)

    overwrite_charge(model, "vacc_c", -1)
    overwrite_charge(model, "vacc_p", -1)
    overwrite_charge(model, "vacc_e", -1)

    overwrite_charge(model, "xylan4_c", -1) # og = 0, no charge given in Bigg, but -1 fits equations
    overwrite_charge(model, "xylan4_e", -1)


    # second: reactions
    overwrite_reaction(model, "3HPAOX", # H was removed from this reaction
                       {"3hoxpac_c": -1.0,
                        "nadh_c": -1.0,
                        "o2_c": -1.0,
                        "34dhpha_c": 1.0,
                        "h2o_c": 1.0,
                        "nad_c": 1.0})

    overwrite_reaction(model, "3SALATAi", # this and ASPA2 are duplicate reactions (only differencs is an H), reaction was curated according to metacyc; https://biocyc.org/reaction?orgid=META&id=3-SULFINOALANINE-AMINOTRANSFERASE-RXN
                       {"3sala_c": -1.0,
                        "akg_c": -1.0,
                        "3snpyr_c": 1.0,
                        "glu__L_c": 1.0})

    overwrite_reaction(model, "6PGALSZ_1", {
        "h2o_c": -1.0,
        "lac6p_c": -1.0,
        "dgal6p_c": 1.0,
        "glc__D_c": 1.0
    })
    if "6PGALSZ_1" in model.reactions and "6PGALSZ" not in model.reactions:
        AA4_curate.reactions.get_by_id("6PGALSZ_1").id = "6PGALSZ"

    overwrite_reaction(model, "ACOAM", # H was removed
                       {"ac_c": -1.0,
                        "atp_c": -1.0,
                        "acadl_c": 1.0,
                        "ppi_c": 1.0})

    overwrite_reaction(model, "ACPS1",
                       {"apoACP_c": -1.0,
                        "coa_c": -1.0,
                        "ACP_c": 1.0,
                        "pap_c": 1.0})

    overwrite_reaction(model, "ACPpds",
                       {"ACP_c": -1.0,
                        "h2o_c": -1.0,
                        "apoACP_c": 1.0,
                        "h_c": 2.0,
                        "pan4p_c": 1.0})

    overwrite_reaction(model, "ALDD31_1",
                       {"gly_c": 1.0,
                        "h_c": 2.0,
                        "h2o_c": -1.0,
                        "nad_c": -1.0,
                        "nadh_c": 1.0,
                        "amacald_c": -1})

    overwrite_reaction(model, "ASR",
                       {"aso4_c": -1.0,
                        "gthrd_c": -2.0,
                        "h_c": -1.0,
                        "aso3_c": 1.0,
                        "gthox_c": 1.0,
                        "h2o_c": 1.0})

    # https://biocyc.org/reaction?orgid=META&id=RXN-10737
    overwrite_reaction(model, "ASR2",
                       {"aso4_c": -1.0,
                        "trdrd_c": -1.0,
                        "h_c": -1.0,
                        "aso3_c": 1.0,
                        "h2o_c": 1.0,
                        "trdox_c": 1.0})

    # was overwritten by BIGG, but before that the H was in the reaction and equals also this reaction BKDC that is e.g. in AA1
    overwrite_reaction(model, "AT_MBD2",
                       {"dhlam_c": -1.0,
                        "ibcoa_c": -1.0,
                        "2mpdhl_c": 1.0,
                        "coa_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "BEF",
                       {"betald_c": -1.0,
                        "fad_c": -1.0,
                        "h2o_c": -1.0,
                        "fadh2_c": 1.0,
                        "glyb_c": 1.0,
                        "h_c": 1.0})

    if "COCO2" not in model.reactions and "CO2FO" in model.reactions:
        overwrite_reaction(model, "CO2FO",
                               {"co_c": -1.0,
                                "fdxox_c": -2.0,
                                "h2o_c": -1.0,
                                "co2_c": 1.0,
                                "fdxrd_c": 2.0,
                                "h_c": 2.0})
        model.reactions.CO2FO.upper_bound = 1000
        model.reactions.CO2FO.lower_bound = -1000
        model.reactions.CO2FO.id = "COCO2"

    overwrite_reaction(model, "CMLDC", # https://modelseed.org/biochem/reactions/rxn02483
                       {"4cml_c": -1.0,
                        "h_c": -1.0, # h changed from product to educt site
                        "5odhf2a_c": 1.0,
                        "co2_c": 1.0})
    if "4CMLCL_kt" not in model.reactions and "CMLDC" in model.reactions:
        rxn = model.reactions.get_by_id("CMLDC")
        rxn.id = "4CMLCL_kt"

    overwrite_reaction(model, "DACL", # https://biocyc.org/reaction?orgid=META&id=RXN0-5040 H was removed
                       {"abg4_c": -1.0,
                        "h2o_c": -1.0,
                        "4abz_c": 1.0,
                        "glu__D_c": 1.0})

    overwrite_reaction(model, "DHBZS2H",
                       {"23dhbzs2_c": -1.0,
                        "h2o_c": -1.0,
                        "h_c": 2.0, # new because of logic
                        "23dhbzs_c": 2.0})

     # https://biocyc.org/reaction?orgid=META&id=RXN-14477
    overwrite_reaction(model, "ENTERH",
                       {"enter_c": -1.0,
                        "h2o_c": -1.0,
                        "23dhbzs3_c": 1.0,
                        "h_c": 1.0}) # h was added

    overwrite_reaction(model, "FADD3",
                       {"atp_c": -1.0,
                        "coa_c": -1.0,
                        "hip_c": -1.0,
                        "hipcoa_c": 1.0,
                        "ppi_c": 1.0,
                        "amp_c": 1.0})

    overwrite_reaction(model, "FE3DHBZS3R",
                       {"fe3dhbzs3_c": -2.0,
                        "nadph_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 3.0,
                        "nadp_c": 1.0})

    overwrite_reaction(model, "FEDHBZS3R1",
                       {"fe3dhbzs3_c": -2.0,
                        "fadh2_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 4.0,
                        "fad_c": 1.0})

    overwrite_reaction(model, "FEDHBZS3R2",
                       {"fe3dhbzs3_c": -2.0,
                        "fmnh2_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 4.0,
                        "fmn_c": 1.0})

    overwrite_reaction(model, "FEDHBZS3R3",
                       {"fe3dhbzs3_c": -2.0,
                        "rbflvrd_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 4.0,
                        "ribflv_c": 1.0})

    overwrite_reaction(model, "FNOR",
                       {"fdxrd_c": -2.0,
                        "h_c": -1.0,
                        "nadp_c": -1.0,
                        "fdxox_c": 2.0, # replaces fdxo_2_2_c
                        "nadph_c": 1.0})

    overwrite_reaction(model, "FORGLUIH2",
                       {"forglu_c": -1.0,
                        "h2o_c": -1.0,
                        "Nforglu_c": 1.0,
                        "nh4_c": 1.0})

    # https://biocyc.org/reaction?orgid=META&id=1.18.1.2-RXN change of stoichiometry
    overwrite_reaction(model, "FPRA",
                       {"fdxrd_c": -2.0,
                        "h_c": -1.0,
                        "nadp_c": -1.0,
                        "fdxox_c": 2.0,
                        "nadph_c": 1.0})

    # https://modelseed.org/biochem/reactions/rxn28276 (immer noch charge imbalance, aber mass stimmt)
    overwrite_reaction(model, "GCDH",
                       {"glutcoa_c": -1.0,
                        "b2coa_c": 1.0,
                        "h_c": 1.0,
                        "co2_c": 1.0})

    # https://metacyc.org/reaction?orgid=META&id=GLUTAMATE-SYNTHASE-FERREDOXIN-RXN#
    overwrite_reaction(model, "GLMS_syn",
                       {"fdxrd_c": -2.0,
                        "akg_c": -1.0,
                        "gln__L_c": -1.0,
                        "h_c": -2.0,
                        "glu__L_c": 2.0,
                        "fdxox_c": 2.0}) # replaces fdxo_2_2_c because we need +2 charge

    overwrite_reaction(model, "GLUTRS_3",
                       {"atp_c": -1.0,
                        "glu__L_c": -1.0,
                        "trnaglu_c": -1.0,
                        "amp_c": 1.0,
                        "glutrna_c": 1.0,
                        "ppi_c": 1.0})

    overwrite_reaction(model, "GLYCS_I",
                       {"gthrd_c": -1.0,
                        "mthgxl_c": -1.0,
                        "lgt__S_c": 1.0}) # og = lgt_s_c; they are duplicates

    overwrite_reaction(model, "GLYCS_II",
                       {"h2o_c": -1.0,
                        "lgt__S_c": -1.0, # og = lgt_s_c; they are duplicates
                        "gthrd_c": 1.0,
                        "h_c": 1.0,
                        "lac__L_c": 1.0})

    overwrite_reaction(model, "GLYTYRabc",
                       {"atp_c": -1.0,
                        "gly_tyr_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_tyr_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GLYLEUtr",
                       {"atp_c": -1.0,
                        "gly_leu_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_leu_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GLYPHEtr",
                       {"atp_c": -1.0,
                        "gly_phe_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_phe_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GLYCYSabc",
                       {"atp_c": -1.0,
                        "gly_cys_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_cys_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GTPDPK_1",
                       {"atp_c": -1.0,
                        "gtp_c": -1.0,
                        "amp_c": 1.0,
                        "gdptp_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "HIPA",
                       {"hip_c": -1.0,
                        "nadh_c": -1.0,
                        "5ohhip_c": 1.0,
                        "nad_c": 1.0})

    overwrite_reaction(model, "HSAC",
                       {"34dhsa_c": -1.0,
                        "o2_c": -1.0,
                        "49dsha_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "IPDAB",
                       {"5ohhip_c": -1.0,
                        "atp_c": -1.0,
                        "coa_c": -1.0,
                        "h_c": -1.0,
                        "5ohhipcoa_c": 1.0,
                        "amp_c": 1.0,
                        "ppi_c": 1.0})

    overwrite_reaction(model, "MECDPDH3_syn",
                       {"2mecdp_c": -1.0,
                        "fdxrd_c": -2.0,
                        "h_c": -1.0,
                        "fdxox_c": 2.0, # replaces fdxo_2_2_c
                        "h2mb4p_c": 1.0,
                        "h2o_c": 1.0})
    overwrite_reaction(model, "MECDPDH4E", # AA3 only has this reaction but not MECDPDH3_syn which in other models are duplicates; to be able to better compare between models, i am going to change the name of the reaction
                       {"2mecdp_c": -1.0,
                        "fdxrd_c": -2.0,
                        "h_c": -1.0,
                        "fdxox_c": 2.0, # replaces fdxo_2_2_c
                        "h2mb4p_c": 1.0,
                        "h2o_c": 1.0})
    if "MECDPDH3_syn" not in model.reactions and "MECDPDH4E" in model.reactions:
        rxn = model.reactions.get_by_id("MECDPDH4E")
        rxn.id = "MECDPDH3_syn"


    overwrite_reaction(model, "MRF", # https://www.genome.jp/entry/R01217, iron complexes were doubled in stoichiometry according to kegg; H was changed in accordance to mass/charge balance and out formulas/charges mlthf/5mthf are diff from kegg
                       {"fdxrd_c": -2.0,
                        "h_c": -3.0,
                        "mlthf_c": -1.0,
                        "5mthf_c": 1.0,
                        "fdxox_c": 2.0})

    overwrite_reaction(model, "MS_1",
                       {"hcys__L_c": -1.0,
                        "mhpglu_c": -1.0,
                        "hpglu_c": 1.0,
                        "met__L_c": 1.0})

    # there is still charge imbalance with his reaction but the fix gets rid of the mass inbalance
    overwrite_reaction(model, "NMO",
               {"etha_c": -1.0,
                "fmnh2_c": -1.0,
                "o2_c": -1.0,
                "acald_c": 1.0,
                "fmn_c": 1.0,
                "no2_c": 1.0,
                "h_c": 6.0  # this was og reaction but was overwritten with bigg info (H was lost)
                })

    overwrite_reaction(model, "OOR3r", # https://biocyc.org/reaction?orgid=META&id=2-OXOGLUTARATE-SYNTHASE-RXN; https://www.genome.jp/dbget-bin/www_bget?ec:1.2.7.3 EC number was given on BIGG page for that reaction but bigg was a bit off
                       {"akg_c": -1.0,
                        "coa_c": -1.0,
                        "fdxox_c": -2.0,
                        "succoa_c": 1.0,
                        "co2_c": 1.0,
                        "h_c": 1.0,
                        "fdxrd_c": 2.0})

    overwrite_reaction(model, "PACPT_1",
                       {"amp_c": 1.0,
                        "coa_c": -1.0,
                        "ppcoa_c": 1.0,
                        "ppad_c": -1.0})

    # https://modelseed.org/biochem/reactions/rxn13395 out scl is only -7, in seed it is -8, so we need only one H
    overwrite_reaction(model, "PC2DHG",
                       {"dscl_c": -1.0,
                        "nadp_c": -1.0,
                        "nadph_c": 1.0,
                        "scl_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "PCADYOX", # https://modelseed.org/biochem/reactions/rxn01192
                       {"34dhbz_c": -1.0,
                        "o2_c": -1.0,
                        "CCbuttc_c": 1.0,
                        "h_c": 2.0}) # was added

    overwrite_reaction(model, "PDIMAS",
                           {"h_c": -1.0, #-2
                            "phthiocerol_c": -1.0,
                            "tamocta_c": -2.0,
                            "h2o_c": 2.0,
                            "pdima_c": 1.0})

    overwrite_reaction(model, "POR_syn",
                       {"fdxox_c": -2.0, # replaces fdxo_2_2_c
                        "coa_c": -1.0,
                        "pyr_c": -1.0,
                        "accoa_c": 1.0,
                        "co2_c": 1.0,
                        "h_c": 1.0,
                        "fdxrd_c": 2.0})

    overwrite_reaction(model, "PRAIS",
                       {"atp_c": -1.0,
                        "fpram_c": -1.0,
                        "adp_c": 1.0,
                        "air_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 2.0})

    overwrite_reaction(model, "QSDH",
                       {"pqq_c": -1.0,
                        "skm_c": -1.0,
                        "3dhsk_c": 1.0,
                        "pqqh2_c": 1.0})

    # Salmochelin fixes (there are first fixes by Frowin in the apply mass balance function notebook)
    overwrite_reaction(model, "SALCHS1H",
                       {"h2o_c": -1.0,
                        "salchs1_c": -1.0,
                        "23dhbzs_c": 1.0,
                        "salchsx_c": 1.0,
                        "h_c": 2.0})
    overwrite_reaction(model, "SALCHS2H",
                       {"h2o_c": -1.0,
                        "salchs2_c": -1.0,
                        "salchs1_c": 1.0,
                        "salchsx_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "SULR_syn",
                       {"fdxrd_c": -6.0,
                        "h_c": -8.0,
                        "so3_c": -1.0,
                        "fdxox_c": 6.0, # fdxo_2_2 was replaced by fdxox_c
                        "h2o_c": 3.0,
                        "h2s_c": 1.0})

    overwrite_reaction(model, "SMIA1",
                       {"fe3_e": -1.0,
                        "stfrnA_e": -1.0,
                        "istfrnA_e": 1.0})

    overwrite_reaction(model, "SMIA1abc",
                       {"atp_c": -1.0,
                        "h2o_c": -1.0,
                        "istfrnB_e": -1.0,
                        "adp_c": 1.0,
                        "fe3_c": 1.0,
                        "h_c": 1.0,
                        "pi_c": 1.0,
                        "stfrnB_c": 1.0})

    overwrite_reaction(model, "SMIA2abc",
                       {"atp_c": -1.0,
                        "h2o_c": -1.0,
                        "istfrnA_e": -1.0,
                        "adp_c": 1.0,
                        "fe3_c": 1.0,
                        "h_c": 1.0,
                        "pi_c": 1.0,
                        "stfrnA_c": 1.0})

    overwrite_reaction(model, "SMIB1",
                       {"fe3_e": -1.0,
                        "stfrnB_e": -1.0,
                        "istfrnB_e": 1.0})

    overwrite_reaction(model, "STAS",
                       {"atp_c": -2.0,
                        "cit_c": -2.0,
                        "orn_c": -1.0,
                        "amp_c": 2.0,
                        "ppi_c": 2.0,
                        "stfrnA_c": 1.0,
                        "h_c": 2.0}) # H changed sides, similar to here https://biocyc.org/reaction?orgid=META&id=RXN-19521 but reaction is still different

    overwrite_reaction(model, "TAGabc",
                           {"atp_c": -1.0,
                            "h2o_c": -1.0,
                            "tagur_e": -1.0,
                            "adp_c": 1.0,
                            "h_c": 1.0,
                            "pi_c": 1.0,
                            "tagur_c": 1.0})

    overwrite_reaction(model, "T6PK",
                       {"atp_c": -1.0,
                        "tag6p__D_c": -1.0,
                        "adp_c": 1.0,
                        "tagdp__D_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "THZSN",
                       {"cys__L_c": -1.0,
                        "dxyl_c": -1.0,
                        "tyr__L_c": -1.0,
                        "h_c": 2.0, # changing sides
                        "4hba_c": 1.0,
                        "4mhetz_c": 1.0,
                        "co2_c": 1.0,
                        "h2o_c": 1.0,
                        "nh4_c": 1.0,
                        "pyr_c": 1.0})

    # https://modelseed.org/biochem/reactions/rxn10816
    overwrite_reaction(model, "THZSN_1",
                       {"cys__L_c": -1.0,
                        "dxyl_c": -1.0,
                        "fdxox_c": -1.0, # instead of fdx_2_2_c
                        "tyr__L_c": -1.0,
                        "4hba_c": 1.0,
                        "4mhetz_c": 1.0,
                        "co2_c": 1.0,
                        "fdxrd_c": 1.0,
                        "h2o_c": 1.0,
                        "h_c": 2.0, # 2 instead of 1;
                        "nh4_c": 1.0,
                        "pyr_c": 1.0})

    if "UNK2" in model.reactions and "UNK3" not in model.reactions:
        overwrite_reaction(model, "UNK2",
                           {"2kmb_c": -1.0,
                            "glu__L_c": -1.0,
                            "akg_c": 1.0,
                            "met__L_c": 1.0})
        model.reactions.UNK2.id = "UNK3" #UNK2 seems to be a lumped reaction that should be UNK3 instead

    # deletions
    # delete_duplicate_reaction(model, "ACPS1_1", "ACPS1", gpr=True) # cannot delete because gprs dont overlap
    # delete_duplicate_reaction(model, "ASPA2", "3SALATAi", gpr=True) # cannot delete because gprs dont overlap
    delete_duplicate_reaction(model, "CMLDC", "4CMLCL_kt", gpr=True)
    delete_duplicate_reaction(model, "CO2FO", "COCO2") #CO2FO is only in 1 model in BIGG
    delete_duplicate_reaction(model, "FMNAT_1", "FMNAT") # same gpr
    delete_duplicate_reaction(model, "GAPD_1", "GAPD", gpr=True) # GAPD_1 gpr is in GAPD
    delete_duplicate_reaction(model, "GTPDPK_1", "GTPDPK", gpr=True) # depending on model, gpr's match
    delete_duplicate_reaction(model, "HMEDS", "MECDPDH3_syn") # both have no gpr
    delete_duplicate_reaction(model, "MECDPDH4E", "MECDPDH3_syn") # same gpr
    delete_duplicate_reaction(model, "PRFGCL", "PRAIS", gpr=True) # same gpr
    delete_duplicate_reaction(model, "RBFK_1", "RBFK") # same gpr
    delete_reaction(model, "FNRR3") # no GPR, only 1 BIGG model, false oxidized ferrodoxin metabolite

    if "PRFGS" in model.reactions and "PRFGS_1" in model.reactions: # gprs overlap but both have something unique thats why special command for that
        model.reactions.get_by_id("PRFGS").gene_reaction_rule = model.reactions.get_by_id("PRFGS").gene_reaction_rule + " or (WP_079211777_1 and WP_079211778_1)"
        delete_reaction(model, "PRFGS_1")

    # all three metabolites are only part of this reaction
    # PENAM: h2o_p + peng_p <=> 6apa_p + pac_p
    if "peng_p" in model.metabolites and "6apa_p" in model.metabolites and "pac_p" in model.metabolites:
        if len(model.metabolites.get_by_id("peng_p").reactions) == 1 and len(model.metabolites.get_by_id("6apa_p").reactions) == 1 and len(model.metabolites.get_by_id("pac_p").reactions) == 1:
            delete_reaction(model, "PENAM")
            delete_metabolite(model, "peng_p")
            delete_metabolite(model, "6apa_p")
            delete_metabolite(model, "pac_p")

    delete_metabolite(model, "fdxo_2_2_c")
    delete_metabolite(model, "lgt_s_c")
    delete_metabolite(model, "glc_D_B_c")



In [295]:
overwrite_manual(AA1_curate)
overwrite_manual(AA2_curate)
overwrite_manual(AA3_curate)
overwrite_manual(AA4_curate)
overwrite_manual(AA5_curate)
overwrite_manual(AA6_curate)
overwrite_manual(AA7_curate)

In [288]:
check_balance(AA1_curate)

There are 4 unbalanced reactions in AA1


{<Reaction AMMQT8 at 0x7075d4e1c430>: {'charge': -2.0},
 <Reaction DHNAOT at 0x7075d5feaaa0>: {'charge': 2.0},
 <Reaction HDECH at 0x7075c5bddb40>: {'charge': 2.0},
 <Reaction NMO at 0x7075c584f0d0>: {'charge': 4.0}}

In [289]:
check_balance(AA2_curate)

There are 6 unbalanced reactions in AA2


{<Reaction AHMMPS at 0x7075c50b0d90>: {'charge': -2.0},
 <Reaction AMMQT8 at 0x7075c51064a0>: {'charge': -2.0},
 <Reaction BTS2 at 0x7075c4f9d990>: {'charge': 2.0},
 <Reaction DHNAOT at 0x7075c4efaa40>: {'charge': 2.0},
 <Reaction GCDH at 0x7075c4b724a0>: {'charge': 2.0},
 <Reaction NMO at 0x7075c46af070>: {'charge': 4.0}}

In [296]:
check_balance(AA3_curate)
# APPATr_BS and G3POA_BS are not imbalanced but because these reactions contain very large metabolites, so there are slight "errors", e.g. imbalance of C -2.3*10^(-15) which is 0

There are 10 unbalanced reactions in AA3


{<Reaction AGPATr_BS at 0x7075e769dc00>: {'charge': 1.6653345369377348e-16,
  'C': -2.220446049250313e-15,
  'H': -5.329070518200751e-15,
  'N': -1.7486012637846216e-15,
  'O': -1.5543122344752192e-15,
  'P': 1.942890293094024e-16,
  'S': -4.163336342344337e-17},
 <Reaction DHNAOT at 0x7075e75ffd00>: {'charge': 2.0},
 <Reaction G3POA_BS at 0x7075ec423010>: {'charge': 1.6653345369377348e-16,
  'C': -2.220446049250313e-15,
  'H': -5.329070518200751e-15,
  'N': -1.7486012637846216e-15,
  'O': 2.220446049250313e-16,
  'P': -2.498001805406602e-16,
  'S': -4.163336342344337e-17},
 <Reaction GCDH at 0x7075e5af95d0>: {'charge': 2.0},
 <Reaction HDECH at 0x7075ec404520>: {'charge': 2.0},
 <Reaction LIPO1S24_BS at 0x7075e66f6e60>: {'charge': 2160.0},
 <Reaction LIPO2S24_BS at 0x7075e66f66b0>: {'charge': 2160.0},
 <Reaction LIPO3S24_BS at 0x7075e66f4730>: {'charge': 2160.0},
 <Reaction LIPO4S24_BS at 0x7075e66f5030>: {'charge': 2160.0},
 <Reaction NMO at 0x7075e5a27520>: {'charge': 4.0}}

In [290]:
check_balance(AA4_curate)
# CITDAPPS, LDPCDES, STFRNS are all connected

There are 6 unbalanced reactions in AA4


{<Reaction CITDAPPS at 0x7075e4fa70d0>: {'H': -1.0},
 <Reaction GCDH at 0x7075e5f07100>: {'charge': 2.0},
 <Reaction LDPCDES at 0x7075e799bca0>: {'charge': 3.0},
 <Reaction STFRNS at 0x7075ec09fe20>: {'C': -3.0, 'O': -7.0},
 <Reaction SUCD2 at 0x7075e8bc79d0>: {'charge': -2.0},
 <Reaction THZSN at 0x7075eae112a0>: {'charge': 2.0}}

In [225]:
get_rxn(AA4_curate, "STFRNS", charge=True, mass=True)
# https://biocyc.org/reaction?orgid=META&id=RXN-19823

STFRNS: akg_c + atp_c + h2o_c + ldpcde_c --> amp_c + h_c + ppi_c + stfrnB_c
{'akg_c': -2, 'amp_c': -2, 'atp_c': -4, 'h_c': 1, 'h2o_c': 0, 'ppi_c': -3, 'ldpcde_c': 0, 'stfrnB_c': -2}
{'akg_c': 'C5H4O5', 'amp_c': 'C10H12N5O7P', 'atp_c': 'C10H12N5O13P3', 'h_c': 'H', 'h2o_c': 'H2O', 'ppi_c': 'HO7P2', 'ldpcde_c': 'C14H18N2O13', 'stfrnB_c': 'C16H22N2O11'}


In [227]:
get_rxn(AA4_curate, "LDPCDES", charge=True, mass=True)

LDPCDES: atp_c + cit_c + citdae_c + h2o_c --> amp_c + h_c + ldpcde_c + ppi_c
{'amp_c': -2, 'atp_c': -4, 'cit_c': -3, 'h_c': 1, 'h2o_c': 0, 'ppi_c': -3, 'citdae_c': 0, 'ldpcde_c': 0}
{'amp_c': 'C10H12N5O7P', 'atp_c': 'C10H12N5O13P3', 'cit_c': 'C6H5O7', 'h_c': 'H', 'h2o_c': 'H2O', 'ppi_c': 'HO7P2', 'citdae_c': 'C8H13N2O6', 'ldpcde_c': 'C14H18N2O13'}


In [233]:
get_rxn(AA4_curate, "CITDAPPS", charge=True, mass=True)
# https://biocyc.org/reaction?orgid=META&id=RXN-19820

CITDAPPS: 23dappa_c + atp_c + cit_c + h_c --> adp_c + citdapp_c + pi_c
{'23dappa_c': 1, 'adp_c': -3, 'atp_c': -4, 'cit_c': -3, 'h_c': 1, 'pi_c': -2, 'citdapp_c': 0}
{'23dappa_c': 'C3H9N2O2', 'adp_c': 'C10H12N5O10P2', 'atp_c': 'C10H12N5O13P3', 'cit_c': 'C6H5O7', 'h_c': 'H', 'pi_c': 'HO4P', 'citdapp_c': 'C9H13N2O8'}


In [291]:
check_balance(AA5_curate)

There are 2 unbalanced reactions in AA5


{<Reaction HDECH at 0x7075e7e57fd0>: {'charge': 2.0},
 <Reaction NMO at 0x7075e66d3bb0>: {'charge': 4.0}}

In [292]:
check_balance(AA6_curate)

There are 4 unbalanced reactions in AA6


{<Reaction AMMQT8 at 0x7075e18386a0>: {'charge': -2.0},
 <Reaction DHNAOT at 0x7075e2cf5720>: {'charge': 2.0},
 <Reaction GCDH at 0x7075e33f8a60>: {'charge': 2.0},
 <Reaction NMO at 0x7075e3e4dea0>: {'charge': 4.0}}

In [293]:
check_balance(AA7_curate)

There are 4 unbalanced reactions in AA7


{<Reaction DHNAOT at 0x7075c3c07340>: {'charge': 2.0},
 <Reaction MECDPDH at 0x7075c3475210>: {'charge': -2.0},
 <Reaction NMO at 0x7075c337b5b0>: {'charge': 4.0},
 <Reaction SUCD2 at 0x7075c2e258d0>: {'charge': -2.0}}

It is not possible to curate all reactions, some are just inherently imbalanced.

In [49]:
# save all curated model as file
for model_name, model in models_curation.items():
    path = f"../Models/02_afterCuration/{model_name[:3]}.xml"
    write_sbml_model(model, path)

## Scale Down Teichoic Acids

In [113]:
# AA3 has some problems with teichoic acids reactions
# i.e. LIPO1S24_BS, LIPO2S24_BS, LIPO3S24_BS, LIPO4S24_BS
check_mass_balance(AA3_curate)

{<Reaction AGPATr_BS at 0x723922631de0>: {'charge': 1.6653345369377348e-16,
  'C': -2.220446049250313e-15,
  'H': -5.329070518200751e-15,
  'N': -1.7486012637846216e-15,
  'O': -1.5543122344752192e-15,
  'P': 1.942890293094024e-16,
  'S': -4.163336342344337e-17},
 <Reaction DHNAOT at 0x723922de7430>: {'charge': 2.0},
 <Reaction G3POA_BS at 0x72392205b220>: {'charge': 1.6653345369377348e-16,
  'C': -2.220446049250313e-15,
  'H': -5.329070518200751e-15,
  'N': -1.7486012637846216e-15,
  'O': 2.220446049250313e-16,
  'P': -2.498001805406602e-16,
  'S': -4.163336342344337e-17},
 <Reaction HDECH at 0x723922fe5a20>: {'charge': 2.0},
 <Reaction LIPO1S24_BS at 0x723922482710>: {'charge': 2160.0},
 <Reaction LIPO2S24_BS at 0x723922482980>: {'charge': 2160.0},
 <Reaction LIPO3S24_BS at 0x723922481de0>: {'charge': 2160.0},
 <Reaction LIPO4S24_BS at 0x723922482c80>: {'charge': 2160.0},
 <Reaction MECDPDH at 0x72392302d900>: {'charge': -2.0},
 <Reaction NMO at 0x7239231be320>: {'charge': 4.0}}

In [297]:
def scale_down_lipo(model):
    # LIPO1S24_BS/LIPO2S24_BS/LIPO3S24_BS/LIPO4S24_BS
    overwrite_reaction(model, "LIPO1S24_BS",{
        "cdpglyc_c": -1.0,
        "d12dg_BS_c": -(1/2400),
        "udpg_c": -1.0,
        "cmp_c": 1.0,
        "h_c": 2.0,
        "lipo1_24_BS_c": (1/2400),
        "udp_c": 1.0
    })
    overwrite_reaction(model, "LIPO2S24_BS",{
        "cdpglyc_c": -1.0,
        "d12dg_BS_c": -(1/2400),
        "uacgam_c": -1.0,
        "cmp_c": 1.0,
        "h_c": 2.0,
        "lipo2_24_BS_c": (1/2400),
        "udp_c": 1.0
    })
    overwrite_reaction(model, "LIPO3S24_BS",{
        "ala__D_c": -1.0,
        "atp_c": -1.0,
        "cdpglyc_c": -1.0,
        "d12dg_BS_c": -(1/2400),
        "h2o_c": -1.0,
        "amp_c": 1.0,
        "cmp_c": 1.0,
        "h_c": 2.0,
        "lipo3_24_BS_c": (1/2400),
        "ppi_c": 1.0
    })
    overwrite_reaction(model, "LIPO4S24_BS",{
        "cdpglyc_c": -1.0,
        "d12dg_BS_c": -(1/2400),
        "cmp_c": 1.0,
        "h_c": 2.0,
        "lipo4_24_BS_c": (1/2400)
    })

In [298]:
scale_down_lipo(AA3_curate)

In [53]:
write_sbml_model(AA3_curate, "../Models/02_afterCuration/AA3.xml")

## Go through unbalanced reactions

In [117]:
inbalanced_rxns = {}
for model in models_curation.values():
    current_imbalances = check_mass_balance(model)
    for rxn, charge in current_imbalances.items():
        if rxn.id != "AGPATr_BS" and rxn.id != "G3POA_BS":
            inbalanced_rxns.update({rxn.id:charge})

In [118]:
print(len(inbalanced_rxns))
inbalanced_rxns

11


{'AMMQT8': {'charge': -2.0},
 'NMO': {'charge': 4.0},
 'DHNAOT': {'charge': 2.0},
 'GCDH': {'charge': 2.0},
 'THZSN_1': {'charge': 1.0},
 'HDECH': {'charge': 2.0},
 'LIPO1S24_BS': {'charge': 0.8999999999999999,
  'C': 1.7763568394002505e-15,
  'H': 3.552713678800501e-15},
 'LIPO2S24_BS': {'charge': 0.8999999999999999,
  'C': 1.7763568394002505e-15,
  'H': 3.552713678800501e-15},
 'LIPO3S24_BS': {'charge': 0.8999999999999999,
  'C': -8.881784197001252e-16,
  'H': -3.552713678800501e-15,
  'O': 8.881784197001252e-16},
 'LIPO4S24_BS': {'charge': 1.9, 'H': 1.0},
 'MECDPDH': {'charge': -2.0}}

In [119]:
for rxn in inbalanced_rxns.keys():
    for model in models_curation.values():
        if rxn in model.reactions:
            print(rxn,"is in", model.id)

AMMQT8 is in AA1
AMMQT8 is in AA2
AMMQT8 is in AA6
NMO is in AA1
NMO is in AA3
NMO is in AA5
NMO is in AA6
NMO is in AA7
DHNAOT is in AA2
DHNAOT is in AA3
DHNAOT is in AA6
DHNAOT is in AA7
GCDH is in AA2
GCDH is in AA4
GCDH is in AA6
THZSN_1 is in AA2
HDECH is in AA3
HDECH is in AA5
LIPO1S24_BS is in AA3
LIPO2S24_BS is in AA3
LIPO3S24_BS is in AA3
LIPO4S24_BS is in AA3
MECDPDH is in AA3


# PART 2

# Macaw
Similar to Memote, macaw is also a software for testing metabolic models regarding duplicates, dead-ends,.. \
see: Moyer, D. C., Reimertz, J., Segrè, D., & Fuxman Bass, J. I. (2025). MACAW: a method for semi-automatic detection of errors in genome-scale metabolic models. Genome Biology 2025 26(1). https://doi.org/10.1186/S13059-025-03533-6

## Redox_pairs, proton_ids, phosphate_ids

In [ ]:
redox_pairs = [
    # NAD(H), NADP(H), FAD(H2), and FMN(H2)
    ('nad_c', 'nadh_c'), ('nadp_c', 'nadph_c'), ('fad_c', 'fadh2_c'),
    ('fmn_c', 'fmnh2_c'),
    # ubiquinone-8 and ubiquinol-8
    ('q8_c', 'q8h2_c'),
    # menaquinone-8 and menaquinol-8
    ('mqn8_c', 'mql8_c'),
    # riboflavin
    ('ribflv_c', 'rbflvrd_c'),
    # glutathione
    ('gthox_c', 'gthrd_c'),
    # glutaredoxin
    ('grxox_c', 'grxrd_c'),
    # thioredoxin
    ('trdox_c', 'trdrd_c'),
    # flavodoxin (reduced and semi-oxidized)
    ('flxso_c', 'flxr_c'),
    # oxygen and hydrogen peroxide
    ('o2_c', 'h2o2_c')
]

# protons in all compartments
proton_ids = ['h_c', 'h_e', 'h_p']

# Phosphate in all compartments
pi_ids = ["pi_c", "pi_p", "pi_e"]
ppi_ids = ["ppi_c", "ppi_p", "ppi_e"]

In [ ]:
def filter_existing_redox_pairs(model, redox_pairs):
    metabolites = {met.id for met in model.metabolites}
    return [pair for pair in redox_pairs if pair[0] in metabolites and pair[1] in metabolites]

## Main
we're doing all macaw Tests beside the dilution test because this test has a very high runtime!

In [ ]:
# Directory in which you want to save the resulting csv files
save_dir = '../Reports/A_Macaw_after_first_charge_balance'

In [ ]:
def do_macaw(model_dir, save_dir):
    for file in (f for f in os.listdir(model_dir) if f.endswith(".xml")):

        model = read_sbml_model(os.path.join(model_dir, file))
        model.solver = "cplex"

        (diphosphate_results) = diphosphate_test(model, ppi_ids, pi_ids)

        # not every redox pair is in every model, so we filter first which redox pairs we actually have
        filtered_pairs = filter_existing_redox_pairs(model, redox_pairs)
        (duplicate_results, duplicate_edge_list) = duplicate_test(model, filtered_pairs, proton_ids)

        (dead_end_results, dead_end_edge_list) = dead_end_test(model)

        macaw_results = duplicate_results.merge(diphosphate_results)
        macaw_results = macaw_results.merge(dead_end_results)

        # save results to csv
        file = f"{file[:3]}_macaw_results.csv"
        macaw_results.to_csv(os.path.join(save_dir, file), index=False)

        print(f"Finished processing {file}")
        print("---------------")

In [ ]:
do_macaw(curation_path, save_dir)

# Automated Duplicate Fixes according to Macaw
Based on our results that we got from macaw, we can use them to eliminate some duplicates that are very obvious. We're basing this especially on the GPRs (gene-protein-rule), i.e. if two reaction are flagged as duplicates and have same gpr (reactions or rather enzymes come from same gene) then we can be sure that these are actually duplicates

## Functions

In [301]:
# Summarizes the test results for the diphophate test throughout all files in the test_results_dir
def create_diphsophate_overview(test_results_dir):

    # Create lists to store the reactions that should be irreversible and those that should be flipped and made irreversible
    irreversible = list()
    flipped_and_irr = list()

    # Iterate through the test results directory and read each CSV file
    for file in os.listdir(test_results_dir):
        if file.endswith(".csv"):
            df = pd.read_csv(test_results_dir+file)

            # Create smaller df that only contain the rows with the columns of interest
            # and put all ids in the corresponding list
            irreversible_df = df[df.iloc[:, 6] == 'should be irreversible']
            irreversible.extend(irreversible_df.iloc[:, 0].tolist())
            flipped_and_irr_df = df[df.iloc[:, 6] == 'should be flipped and made irreversible']
            flipped_and_irr.extend(flipped_and_irr_df.iloc[:, 0].tolist())

    # Remove duplicates from the lists
    irreversible = list(set(irreversible))
    flipped_and_irr = list(set(flipped_and_irr))

    # Print the results
    print(f"{len(irreversible)} reactions should be Irreversible reactions:")
    for i in irreversible:
        print(i)
    print("-------------------------------------------------------------------------\n")
    print(f"{len(flipped_and_irr)} reactions should be Flipped and irreversible reactions:")
    for i in flipped_and_irr:
        print(i)

In [ ]:
# A function to create singular reaction pairs in all duplicate lists
# This is necessary since some reactions have several duplicate reactions

# The function takes a list of tuples as input
def formate_duplicate_pairs(duplicate_list):

    # Create a new duplicate list for the formated results at the end = output list
    formated_duplicate_list = list()

    # Iterate through all pairs in the input list and check if there are multiple duplicates for the same reaction
    # This is indicated by the presence of a semicolon
    for r1, r2 in duplicate_list:

        # If a semicolon is found new pairs will be created and added to the output list
        if ";" in r2:

            # Split the string by semicolon and create new pairs
            r2_list = r2.split(";")

            # Create a list for new pairs
            new_pairs = list()

            # Create new pairs from all duplicates and add them to the new_pairs list
            # The new pairs will be sorted to later remove doubles
            for r in r2_list:
                new_pairs.append(tuple(sorted((r1, r))))

            # Extend the output list with new pairs
            formated_duplicate_list.extend(new_pairs)

        # When no semicolon is found the pair is sorted and just added to the output list
        else:
            formated_duplicate_list.append(tuple(sorted((r1, r2))))

    # Remove all duplucate pairs (so all pairs of duplicates that are in the list multiple times)
    formated_duplicate_list = list(set(formated_duplicate_list))

    # Return the formated list of duplicates
    # This is also a list of tuples and most likely longer than the input
    return formated_duplicate_list

In [ ]:
# Use the .csv files from the first MACAW tests to create a list of all 'duplicate pairs' for a model;
# This will be four lists of tuples that represent all reactions in the model that where flagged by;
# MACAW has an exact-, directional-, coefficient- or redox-duplicate

# This takes the model_id as input (without any format suffix)
def get_all_duplicate_pairs(model_id):

    # make sure the files in the test_results_dir correspond with the below naming convention
    test_results = pd.read_csv(test_results_dir+f"{model_id[:3]}_macaw_results.csv")

    # Filter reactions with entries not equal to "ok" in the specified columns
    duplicates_df = test_results.loc[
        (test_results['duplicate_test_exact'] != 'ok') |
        (test_results['duplicate_test_directions'] != 'ok') |
        (test_results['duplicate_test_coefficients'] != 'ok') |
        (test_results['duplicate_test_redox'] != 'ok'),
        ['reaction_id', 'duplicate_test_exact', 'duplicate_test_directions', 'duplicate_test_coefficients', 'duplicate_test_redox']
    ]

    # Create a dict containing the model reactions as keys and the test results for each test as value (dict)
    # This will contain all reactions that were flagged by at least one duplicate test
    model_duplicates_dict = {
        row['reaction_id']: {
            "Exact": row['duplicate_test_exact'],
            "Directions": row['duplicate_test_directions'],
            "Coefficient": row['duplicate_test_coefficients'],
            "Redox": row['duplicate_test_redox']
        }
        for _, row in duplicates_df.iterrows()
    }

    # Create lists for all types of duplicates
    exact_duplicates = list()
    directional_duplicates = list()
    coefficient_duplicates = list()
    redox_duplicates = list()

    # Iterate through all reactions and corresponding duplicate test results
    for reaction_id, tests in model_duplicates_dict.items():

        # Append all reactions flagged by MACAW and the corresponding duplicate reactions
        if tests['Exact'] != 'ok':
            exact_duplicates.append((reaction_id, tests['Exact']))
        if tests['Directions'] != 'ok':
            directional_duplicates.append((reaction_id, tests['Directions']))
        if tests['Coefficient'] != 'ok':
            coefficient_duplicates.append((reaction_id, tests['Coefficient']))
        if tests['Redox'] != 'ok':
            redox_duplicates.append((reaction_id, tests['Redox']))


    # Apply the format_duplicate_pairs function to split reactions with multiple duplicates for the same type
    # into singular pairs (see function below)
    exact_duplicates = formate_duplicate_pairs(exact_duplicates)
    directional_duplicates = formate_duplicate_pairs(directional_duplicates)
    coefficient_duplicates = formate_duplicate_pairs(coefficient_duplicates)
    redox_duplicates = formate_duplicate_pairs(redox_duplicates)

    # Returns all 4 lists
    return exact_duplicates, directional_duplicates, coefficient_duplicates, redox_duplicates

In [ ]:
# Creates a dictionary containing all reactions in the model as keys
# The value is always a list of the sorted genes corresponding to this reaction
# OR ["None"] if no GPR could be found for this reaction in this model

# Input is a cobra model object
def create_gpr_list_dir(model):

    # Create the output dict
    gpr_list_dir = {}

    # Iterate through all reactions in the model
    for reaction in model.reactions:

        # Get the GPR as string
        gpr_string = reaction.gene_reaction_rule

        # If no GPR is given set ["None"] as the value in the output dict
        if gpr_string == '':
            gpr_list_dir[reaction.id] = ["None"]

        # If GPR are found formate them into a list by first splitting them by the 'or' relation
        # Genes with an 'and' relation will be split further and added to the GPR list as an list themself
        else:
            gpr_list_or_split = list(gpr_string.split(' or '))

            # Create a list where both 'or' and 'and' relations are kept = Value list
            gpr_list_or_and_split = []

            # Check if an entry in the list contains an 'and'
            for entry in gpr_list_or_split:

                # If yes the split this entry into a new list, clean it and add the list to the value list
                if 'and' in entry:
                    new_entry = list(entry.split(' and '))
                    new_entry_cleaned = [s.replace("(", "").replace(")", "") for s in new_entry]

                    # Sort this sublist alphabetically
                    gpr_list_or_and_split.append(sorted(new_entry_cleaned))

                # If no then just append the entry into the output list
                else:
                    gpr_list_or_and_split.append(entry)

            # Sort the output list alphabetically by treating each entry as a string
            # Add the sorted value list to the output dict
            gpr_list_dir[reaction.id] = sorted(gpr_list_or_and_split, key=lambda x: ' '.join(x) if isinstance(x, list) else x)

    # Output is a dict with an entry for each reaction in the input model
    return gpr_list_dir

In [ ]:
# A function to compare GPR lists created by the create_gpr_list_dir function
# The output is highly variable and specific
# Only strings describing the relation of the GPR for both input reactions are returned

# Input are two reaction_ids
def compare_gpr_lists(r1, r2):

    # GPR lists for both input reactions are fetched from the dir
    r1_gpr = gpr_list_dir[r1]
    r2_gpr = gpr_list_dir[r2]

    # If either of the reactions has no GPR it is checked if they are both boundary reactions or not
    if (r1_gpr[0] == "None") or (r2_gpr[0] == "None"):
        if (model.reactions.get_by_id(r1) in model.boundary) and (model.reactions.get_by_id(r2) in model.boundary):
            return 'No GPR and both boundary reactions'
        else:
            return 'No GPR but one not a boundary reaction'

    # Do they have the same GPR list?
    elif r1_gpr == r2_gpr:
        return 'Same GPR list'

    # Is the list the same lenght and shares genes or not?
    elif len(r1_gpr) == len(r2_gpr):
        if any(item in r1_gpr for item in r2_gpr):
            return 'Same length of GPR list and share some genes'
        else:
            return 'Same length of GPR list but no common genes'

    # Are the lists of different lenght and:
    # Is one list a sublist of the other?
    # Share the lists genes or not?
    else:
        if (len(r1_gpr) > len(r2_gpr)) and all(item in r1_gpr for item in r2_gpr):
            return 'r2 in r1'
        elif (len(r2_gpr) > len(r1_gpr)) and all(item in r2_gpr for item in r1_gpr):
            return 'r1 in r2'
        elif any(item in r1_gpr for item in r2_gpr):
            return 'Different lenght of GPR list and share some genes'
        else:
            return 'Different lenght of GPR list but no common genes'


In [ ]:
# Path to your models
model_dir = '../Models/02_afterCuration/'

# Path to the .csv files created by the first MACAW run
test_results_dir = '../Reports/A_Macaw_after_first_charge_balance/'

# Path where the new, fixed models (xml files) should be stored
fixed_model_dir = '../Models/02_afterCuration/'

In [ ]:
# Exact duplicates will be removed or combined based on their GPR

def fix_exact_duplicates():

    # Compare the GPR lists of both reactions and apply fixes depending on the returned status
    for r1, r2 in exact_duplicates:
        status = compare_gpr_lists(r1, r2)

        # Delete one reaction if they have the same GPR
        if status == "Same GPR list":
            model.remove_reactions([model.reactions.get_by_id(r2)])

        # If the GPR list of r2 is a substring of the r1 GPR list, delete r2
        elif status == "r2 in r1":
            model.remove_reactions([model.reactions.get_by_id(r2)])

        # If the GPR list of r1 is a substring of the r2 GPR list, delete r1
        elif status == "r1 in r2":
            model.remove_reactions([model.reactions.get_by_id(r1)])

        # If both reactions share genes in their GPR list then combine the the reactions by
        # adding all GPR of r2 to r1 and set the new list as the gpr string for r1
        # Then delete r2
        elif (status == 'Same length of GPR list and share some genes') or (status == 'Different length of GPR list and share some genes'):
            new_gpr_list = list(set(gpr_list_dir[r1].extend(gpr_list_dir[r2])))
            new_gpr_string = create_gpr_strings(new_gpr_list)
            model.reactions.get_by_id(r1).gene_reaction_rule = new_gpr_string
            model.remove_reactions([model.reactions.get_by_id(r2)])

        # For all status where no fix was found yet
        else:
            print(f'exact: No fix possible for {r1} and {r2} with status: {status}')
            continue

        print(f"exact: Fixed reactions {r1} and {r2} with status {status}")


In [ ]:
# A function to transform a GPR list back into a GPR string
def create_gpr_strings(gpr_list):
    result = []
    for item in gpr_list:
        if isinstance(item, list):
            # Join sublist with ' and ' and wrap in parentheses
            sub_str = ' and '.join(item)
            result.append(f'({sub_str})')
        else:
            result.append(item)
    return ' or '.join(result)

In [ ]:
# Directional duplicates will be removed or kept based on their GPR
def fix_directional_duplicates():

    # Compare the GPR lists of both reactions and apply fixes depending on the returned status
    for r1, r2 in directional_duplicates:
        status = compare_gpr_lists(r1, r2)

        # Delete one reaction based on the directionality of both if the GPR list is the same
        # Always keep the reversible reaction or make r1 reversible if both are irreversible and delete r2
        if status == "Same GPR list":
            if model.reactions.get_by_id(r1).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r2)])
            elif model.reactions.get_by_id(r2).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r1)])
            else:
                model.reactions.get_by_id(r1).bounds = (-1000.0, 1000.0)
                model.remove_reactions([model.reactions.get_by_id(r2)])

        # not all possible cases are checked here

        # If the GPR list of r2 is a substring of the r1 GPR list and r1 is reversible, delete r2
        elif status == "r2 in r1":
            if model.reactions.get_by_id(r1).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r2)])

        # If the GPR list of r1 is a substring of the r2 GPR list and r2 is reversible, delete r1
        elif status == "r1 in r2":
            if model.reactions.get_by_id(r2).reversibility:
                model.remove_reactions([model.reactions.get_by_id(r1)])

        # For all status where no fix was found yet
        else:
            print(f'direct: No fix possible for {r1} and {r2} with status: {status}')
            continue

        print(f"direct: Fixed reactions {r1} and {r2} with status {status}")

In [ ]:
# MACAW flags all diphosphate containing reactions that are reversible, since these are typically irreversible.
# After assessing the flagged reactions provided by the create_diphsophate_overview() function (cell above),
# this function sets new bounds for the flagged reactions based on their assessment
def fix_diphophate_reactions():

    make_irreversible_ppi = ['APAT_1','FACOAL140_ISO','NAPRT','NNATr','FACOAL180_2','FACOAL170_anteiso','FACOAL170_ISO','FORMCOAL','FACOAL150_ISO','FACOAL160_ISO','SERASr','OXACOAL','FACOALPREPH','FACOAL150_anteiso','FACOAL200','AADb','UDPACGLP','FACOALPHDCA']
    flip_and_make_irreversible_ppi = ['ORPT']
    stays_reversible = ['GALT','ADK2_1','ADK2']

    # Contrain all reactions that should be irreversible
    for reaction in make_irreversible_ppi:
        try:
            model.reactions.get_by_id(reaction).bounds = (0.0, 1000.0)
            print(f'Fixed {reaction} in {model}')
        except KeyError:
            continue

    # Constrain all reactions that should be irreversible with the diphosphat being a reactant (on the left side in the reaction string)
    for reaction in flip_and_make_irreversible_ppi:
        try:
            model.reactions.get_by_id(reaction).bounds = (-1000.0, 0.0)
            print(f'Fixed {reaction} in {model}')
        except KeyError:
            continue


## Diphosphate test summary
This gives you an overview for all reactions flagged by the diphosphate test throughout all your models

In [ ]:
create_diphsophate_overview(test_results_dir)


## Duplicate Main

In [302]:
# Iterate over all xml files in the model_dir and applies all functions
for file in (f for f in os.listdir(model_dir) if f.endswith(".xml")):

    # Remove the .xml suffix to generate only the model id
    model_id = file[:-4]

    # Prints the model that is investigated right now
    print(f"Model: {model_id}")

    # Parse the model
    model = read_sbml_model(model_dir+file)

    # Create duplicate lists for all four types of duplicates MACAW tests for
    exact_duplicates, directional_duplicates, coefficient_duplicates, redox_duplicates = get_all_duplicate_pairs(model_id)

    # Creates a dict containing the GPR information for all reactions in the model
    gpr_list_dir = create_gpr_list_dir(model)

    # Apply fixes for Diphosphate reactions
    fix_diphophate_reactions()

    # Apply fixes for exact duplicates
    fix_exact_duplicates()

    # Apply fixes for directional duplicates
    fix_directional_duplicates()

    print("-----------------")

    #save model
    write_sbml_model(model, fixed_model_dir+f"{model_id}.xml")

## New Macaw Reports after our fixes


In [ ]:
# Directory where models are
model_dir = "../Models/02_afterCuration/"

# Directory in which you want to save the resulting csv files
save_dir = '../Reports/B_Macaw_after_Duplicate_removing'

In [ ]:
do_macaw(model_dir, save_dir)

# Manual Metabolite Duplicate Removal

## Load Models and Macaw Results

In [ ]:
# Load SBML Models
models_path = "../Models/02_afterCuration/"
models_curation = {}
for model_name in (f for f in os.listdir(models_path) if f.endswith(".xml")):
    model = read_sbml_model(f"{models_path}/{model_name}")
    model.solver = "cplex"
    name = str(model_name[:3]+"_curate")
    models_curation[name] = model

models_curation = {key: models_curation[key] for key in sorted(models_curation.keys())}  # sorts the dictionary alphabetically
AA1_curate, AA2_curate, AA3_curate, AA4_curate, AA5_curate, AA6_curate, AA7_curate = [models_curation[f"AA{i}_curate"] for i in range(1, 8)]

In [ ]:
# Load macaw Results
report_path = "/../Reports/B_Macaw_after_Duplicate_removing/"
macaw = {}

for file in os.listdir(report_path):
    if file.endswith('.csv') and file.startswith('AA'):
        file_path = os.path.join(report_path, file)
        df_name = file.replace('.csv', '')
        macaw[df_name] = pd.read_csv(file_path)

macaw = {key: macaw[key] for key in sorted(macaw.keys())}
AA1_macaw, AA2_macaw, AA3_macaw, AA4_macaw, AA5_macaw, AA6_macaw, AA7_macaw = [macaw[f"AA{i}_macaw_results"] for i in range(1, 8)]

## Functions

In [ ]:
def check_delete(model, compare_list):
    if len(compare_list) != 2:
        print("Please provide exactly two reactions to compare")
        return

    if compare_list[0] in model.reactions and compare_list[1] in model.reactions:
        rxn1 = model.reactions.get_by_id(compare_list[0])
        rxn2 = model.reactions.get_by_id(compare_list[1])

        if rxn1.gene_reaction_rule == rxn2.gene_reaction_rule:
            print(f"Same GPR for both reactions: {rxn1.gene_reaction_rule}")
        else:
            print(f"different GPR for both reactions: {rxn1.gene_reaction_rule} for {compare_list[0]} vs. {rxn2.gene_reaction_rule} for {compare_list[1]}")
    else:
        print("Not all reactions are in the model")

In [ ]:
def extract_met_info_model(model):
    met_infos = []

    for met in model.metabolites:
        met_infos.append({
            "bigg_id": met.id.rsplit("_", 1)[0],  # strip compartment so that it matches the actual BIGG ID that also doesn't have compartments (e.g., glc__D_c to glc__D)
            "model_id": met.id,
            "model_formula": met.formula,
            "model_charge": met.charge
        })

    met_infos = pd.DataFrame(met_infos)
    return met_infos

## Fix Metabolite Duplicates
Using the soft constraints with carveme is the automated and more clever solution to get rid of obvious duplicates that were produced by models in BIGG with an unsatisfactory curation.
However, before moving to using these soft constraints, we did manual duplicate removal. Because there are still a few duplicates in the model that are not only caused by iYO844, the following script is still worth to apply to our soft constrained models.

In [ ]:
def parse_gpr(gpr_str):
    # Split on ' or ' but keep "A and B" as one element
    parts = re.split(r'\s+or\s+', gpr_str)
    # Strip and unify whitespace
    parts = [' '.join(p.strip().split()) for p in parts if p.strip()]
    return set(parts)


def merge_gpr(del_rxn_gpr, keep_rxn_gpr):

    if del_rxn_gpr == "" and keep_rxn_gpr == "":
        return ""

    elif del_rxn_gpr == "":
        return keep_rxn_gpr
    elif keep_rxn_gpr == "":
        return del_rxn_gpr

    else:
        del_rxn_gpr = parse_gpr(del_rxn_gpr)
        keep_rxn_gpr = parse_gpr(keep_rxn_gpr)

        merged_gpr = sorted(del_rxn_gpr.union(keep_rxn_gpr))  # Sort for consistency

        return ' or '.join(merged_gpr)

In [ ]:
def fix_metabolite_duplicates(model, met_delete, met_keep, rxn_dup_pairs,):

    # (0) check if met_delete is in current model and rxn_dup_pairs has right format
    if rxn_dup_pairs != [] and not all(isinstance(pair, tuple) and len(pair) == 2 for pair in rxn_dup_pairs):
        print(f"check rxn_dup_pairs for correctness for {rxn_dup_pairs}")

    elif met_delete in model.metabolites:
        # (1) delete all rxns from met_delete that are in rxn_dup_pairs if met_delete in model
        rxns_met_delete = [rxn[0] for rxn in rxn_dup_pairs if rxn[0] in model.reactions] # all reactions that we need to delete (if they are in the current model)
        rxns_met_keep = [rxn[1] for rxn in rxn_dup_pairs if rxn[1] in model.reactions]

        for dup_pair in rxn_dup_pairs:
            del_rxn, keep_rxn = dup_pair

            # (1.1) if both rxns of one rxn_dup_pairs in model
            if del_rxn in rxns_met_delete and keep_rxn in rxns_met_keep:
                # (1.1.1) merge gpr onto rxn_keep
                del_rxn_gpr = model.reactions.get_by_id(del_rxn).gene_reaction_rule
                keep_rxn_gpr = model.reactions.get_by_id(keep_rxn).gene_reaction_rule
                model.reactions.get_by_id(keep_rxn).gene_reaction_rule = merge_gpr(del_rxn_gpr, keep_rxn_gpr)

                # (1.1.2) delete del_rxn from model
                model.remove_reactions([model.reactions.get_by_id(del_rxn)])

            # (1.2) if only keep_rxn in model, continue with next rxn
            elif keep_rxn in rxns_met_keep:
                continue

            # (1.3) if only del_rxn in model
            elif del_rxn in rxns_met_delete:
                del_rxn_model = model.reactions.get_by_id(del_rxn)
                met_delete_model = model.metabolites.get_by_id(met_delete)
                if met_delete_model in del_rxn_model.metabolites: # should always be the case
                    # (1.3.1) replace met_delete with met_keep in del_rxn
                    coeff = del_rxn_model.get_coefficient(met_delete) # save stoichiometry/coefficient of met_delete

                    # if met_keep doesnt exist in the model, make copy of met_delete but give it ID of met_keep
                    # i cannot just change the id of met_delete because then (3) doesnt work so universally anymore
                    if met_keep not in model.metabolites:
                        original_met = model.metabolites.get_by_id(met_delete)
                        met_keep_model = original_met.copy()
                        met_keep_model.id = met_keep
                        model.add_metabolites([met_keep_model])

                    del_rxn_model.add_metabolites({met_keep: coeff}) # use that coefficient to add met_keep
                    del_rxn_model.add_metabolites({met_delete: -coeff}) # delete met_delete from reaction

                    # (1.3.2) replace id del_rxn with id keep_rxn
                    del_rxn_model.id = keep_rxn

            else:
                #print(f"Problems regarding the duplicate pair: {dup_pair} in model {model.id}. Check manually.")
                # both reactions are not in model, so we can continue to next pair
                continue

        # (2) check if met_delete has rxns left (these are the ones without duplicates that we want to keep)
        left_rxns = [rxn.id for rxn in model.metabolites.get_by_id(met_delete).reactions if rxn.id not in rxns_met_delete]

        # (2.1) change met_delete to met_keep in left_rxns
        for rxn in left_rxns:
            rxn_model = model.reactions.get_by_id(rxn)
            coeff = rxn_model.get_coefficient(met_delete)

            # if met_keep doesnt exist in the model, make copy of met_delete but give it ID of met_keep
            # i cannot just change the id of met_delete because then (3) doesnt work so universally anymore
            if met_keep not in model.metabolites:
                original_met = model.metabolites.get_by_id(met_delete)
                met_keep_model = original_met.copy()
                met_keep_model.id = met_keep
                model.add_metabolites([met_keep_model])

            rxn_model.add_metabolites({met_keep: coeff})
            rxn_model.add_metabolites({met_delete: -coeff})

        # (3) delete met_delete
        if len(model.metabolites.get_by_id(met_delete).reactions) == 0:
            met = model.metabolites.get_by_id(met_delete)
            model.metabolites.remove(met)
        else:
            print(f'metabolite {met_delete} cannot be deleted from {model.id} because of reaction(s): {model.metabolites.get_by_id(met_delete).reactions}')


In [ ]:
# Fix and evaluate string from the CSV file below
# convert strings that i get from csv file to lists and add " " around actual strings
def fix_and_parse(s):
    if not isinstance(s, str) or s.strip() == "[]":
        return []

    # Add quotes around unquoted reaction names like BTS, ALCD4, 34DHPACDO
    s_fixed = re.sub(r'(?<![\'"])\b([\w\d]+)\b(?![\'"])', r"'\1'", s)

    try:
        return ast.literal_eval(s_fixed)
    except Exception as e:
        print(f"Error parsing: {s}\nFixed to: {s_fixed}\nError: {e}")
        return []

In [ ]:
def overwrite_part2(model):
    overwrite_reaction(model, "HMEDS", {
        "2mecdp_c": -1.0,
        "fdxrd_c": -2.0,
        "h_c": -1.0,
        "fdxox_c": 2.0,
        "h2o_c": 1.0,
        "h2mb4p_c": 1.0
    })

    overwrite_reaction(model, "ARD_1", {
        "dhmtp_c": -1.0,
        "o2_c": -1.0,
        "2kmb_c": 1.0,
        "for_c": 1.0,
        "h_c": 2.0
    })


    # HMEDS is duplicate and was deleted in 02_ but AA3 only has this reaction but we want to have similar models, so I am renaming the reaction
    if "HMEDS" in model.reactions and "MECDPDH3_syn" not in model.reactions:
        model.reactions.HMEDS.id = "MECDPDH3_syn"

    delete_reaction(model, "HMEDR") # duplicate of IPDPS
    delete_metabolite(model, "hmbpp_c")

In [ ]:
# Load CSV with duplicate info
# this file was created outside of these notebooks, we manually checked metabolites and their reactions to decide if we want to merge metabolites
# the csv consists of 3 columns: (1) metabolite that is obsolete (will be removed), (2) the "right" metaboolite that we're going to keep and (3) a list of tuples where each tuples contains two reactions. the first reaction is the duplicated reaction that comes from the obsolete metabolite and the second reaction is the one we're gonna keep
metabolite_dups = pd.read_csv("../Datasets/Duplicate_Investiagtion/Metabolite_Duplicates.csv", sep=";")

# Change strings from last col to list
col = "rxn_dup_pairs[del_rxn, keep_rxn]"
metabolite_dups[col] = metabolite_dups[col].apply(fix_and_parse)

In [ ]:
# MAIN
for model in models_curation.values(): # go over every model
    for index, row in metabolite_dups.iterrows(): # go over every duplicate metabolite
        met_delete = row["met_delete"]
        met_keep = row["met_keep"]
        rxn_pairs = row["rxn_dup_pairs[del_rxn, keep_rxn]"]

        # Fix the current metabolite
        fix_metabolite_duplicates(model, met_delete, met_keep, rxn_pairs)

    # last minute manual duplicate fixes
    overwrite_part2(model)

In [ ]:
# save all curated model as file
for model_name, model in models_curation.items():
    path = f"../Models/02_afterCuration/{model_name[:3]}.xml"
    write_sbml_model(model, path)

# Check Reaction Duplicates
We already used the Macaw results to fix some exact and directional duplicates if the GPR of the reactions allows us to merge them.
So in the following part of the script, this is more an inspection of all the left-over duplicates rather than an approach to fix them.

## Macaw Exact Duplicates

In [ ]:
def get_macaw_exactdup(macaw_df):
    # Step 1: Filter and get the first and third columns
    pairs = macaw_df[macaw_df["duplicate_test_exact"] != "ok"].iloc[:, [0, 2]].values.tolist()

    # Step 2: Sort each pair (so [A, B] and [B, A] are treated the same)
    sorted_pairs = [sorted(pair) for pair in pairs]

    # Step 3: Remove duplicates by converting to a set of tuples, then back to list
    rxndup_macaw_exact = list(set(tuple(pair) for pair in sorted_pairs))
    return rxndup_macaw_exact


In [ ]:
macaw_exact_duplicates = {f"AA{i}_rxndup_macaw_exact": get_macaw_exactdup(macaw[f"AA{i}_macaw_results"]) for i in range(1, 8)}
AA1_rxndup_macaw_exact, AA2_rxndup_macaw_exact, AA3_rxndup_macaw_exact, AA4_rxndup_macaw_exact, AA51_rxndup_macaw_exact, AA6_rxndup_macaw_exact, AA7_rxndup_macaw_exact = [macaw_exact_duplicates[f"AA{i}_rxndup_macaw_exact"] for i in range(1, 8)]

In [ ]:
combined_exact = list(chain(*macaw_exact_duplicates.values()))
combined_exact= sorted(list(set(combined_exact)))

In [ ]:
print(len(combined_exact))

In [ ]:
combined_exact

In [ ]:
# the remaining reactions have different gpr's that cant be merged (as expected because we fixed everything that is possible with the macaw fixes script)
AA1_rxndup_macaw_exact

## Macaw Directionality Duplicates

In [ ]:
def get_macaw_directdup(macaw_df):
    # Step 1: Filter and get the first and third columns
    pairs = macaw_df[macaw_df["duplicate_test_directions"] != "ok"].iloc[:, [0, 3]].values.tolist()

    # Step 2: Sort each pair (so [A, B] and [B, A] are treated the same)
    sorted_pairs = [sorted(pair) for pair in pairs]

    # Step 3: Remove duplicates by converting to a set of tuples, then back to list
    rxndup_macaw_direct = list(set(tuple(pair) for pair in sorted_pairs))
    return rxndup_macaw_direct

In [ ]:
macaw_directionality_duplicates = {f"AA{i}_rxndup_macaw_direct": get_macaw_directdup(macaw[f"AA{i}_macaw_results"]) for i in range(1, 8)}

AA1_rxndup_macaw_direct, AA2_rxndup_macaw_direct, AA3_rxndup_macaw_direct, AA4_rxndup_macaw_direct, AA51_rxndup_macaw_direct, AA6_rxndup_macaw_direct, AA7_rxndup_macaw_direct = [macaw_directionality_duplicates[f"AA{i}_rxndup_macaw_direct"] for i in range(1, 8)]

In [ ]:
combined_direct = list(chain(*macaw_directionality_duplicates.values()))
combined_direct = sorted(list(set(combined_direct)))
print(len(combined_direct))

# Check Dead-Ends
Dead-ends are a topic for further investigation. Because of the high number of dead-ends it is not possible to solve them manually in a time-efficient way.

In [ ]:
# Basic Counter that counts every occurence of a value in the dead_end_test col of all my results
value_counts = Counter()

# Iterate through each DataFrame in the dictionary
for df in macaw.values():
    if "dead_end_test" in df.columns:
        # Dropna to avoid counting NaN values, optional
        for cell in df["dead_end_test"].dropna():
            # Split by ';', strip whitespace, and update the counter
            values = [val.strip() for val in str(cell).split(';') if val.strip()]
            value_counts.update(values)

# Convert to DataFrame if needed
result_df = pd.DataFrame.from_dict(value_counts, orient='index', columns=['count'])

In [ ]:
result_df

In [ ]:
# concatenate all the results from the dead-end test into one df
# Get unique reaction IDs from all DataFrames
reaction_ids = set(pd.concat([df["reaction_id"] for df in macaw.values()]))

# Create result df where each col is the dead end test from each model
all_dead_ends = pd.DataFrame({"Reaction_ID": sorted(list(reaction_ids))})
all_dead_ends.set_index("Reaction_ID", inplace=True)

# Loop over dictionary items aka each macaw df
for label, df in macaw.items():
    mapping = df.set_index("reaction_id")["dead_end_test"]
    all_dead_ends[label[:3]] = all_dead_ends.index.map(mapping)


In [ ]:
all_dead_ends.to_csv("../Reports/B_Macaw_after_Duplicate_removing/dead_ends_macaw.csv", index=True,na_rep='-')

In [ ]:
all_dead_ends

In [ ]:
# we want to exclude rows where all entries in those columns are "ok" or NaN
# all_dead_ends.iloc[:, 1: ] selects all columns except "Reaction_ID"
mask = (all_dead_ends.iloc[:, 1:] == "ok") | all_dead_ends.iloc[:, 1:].isna()

# Tidentify rows where all entries in those columns match the mask
rows_to_drop = mask.all(axis=1)

# filter them out
filtered_dead_ends = all_dead_ends[~rows_to_drop]

In [ ]:
filtered_dead_ends.to_csv("../Reports/B_Macaw_after_Duplicate_removing/filtered_dead_ends_macaw.csv", index=True,na_rep='-')

In [ ]:
filtered_dead_ends

In [ ]:
dfs = [AA1_macaw, AA2_macaw, AA3_macaw, AA4_macaw, AA5_macaw, AA6_macaw, AA7_macaw]

def normalize_list_merge(val1, val2):
    items = []

    for val in [val1, val2]:
        if val is None or (isinstance(val, float) and pd.isna(val)):
            continue
        if isinstance(val, list):
            items.extend(val)
        else:
            items.append(str(val).strip().lower())

    unique_sorted = sorted(set(items))
    return unique_sorted if unique_sorted else None

def merge_dead_end(df1, df2):
    return pd.merge(
        df1[['reaction_id', 'dead_end_test']],
        df2[['reaction_id', 'dead_end_test']],
        on='reaction_id',
        how='outer',
        suffixes=('_left', '_right')
    ).assign(
        dead_end_test=lambda x: x.apply(
            lambda row: normalize_list_merge(row['dead_end_test_left'], row['dead_end_test_right']),
            axis=1
        )
    )[['reaction_id', 'dead_end_test']]

# Perform the merge
merged_df = reduce(merge_dead_end, dfs)

# Convert list to tuple for counting
merged_df['dead_end_test_tuple'] = merged_df['dead_end_test'].apply(
    lambda x: tuple(x) if isinstance(x, list) else x
)

# Count occurrences
dead_end_counts = merged_df['dead_end_test_tuple'].value_counts(dropna=False)

dead_end_counts

In [ ]:
# Convert to tuple to make it hashable
merged_df['dead_end_test_tuple'] = merged_df['dead_end_test'].apply(
    lambda x: tuple(x) if isinstance(x, list) else x
)

dead_end_counts = merged_df['dead_end_test_tuple'].value_counts(dropna=False)
dead_end_counts


# Gap-Filling
There are issues that some models not show a positive growth phenotype, which needs to be investigated.

## Step-by-step investigation of biomass (objective)

In [ ]:
# this function is specifically for the Biomass/Growth reaction which is the objective of all models; it checks if those compounds can be produced iff they were the objective
def check_every_biomass_component(model, medium_dict):
    rxn_id = "Growth_small"
    sad_mets = []

    growth_mets = model.reactions.Growth.metabolites
    for met, flux in growth_mets.items():
        if flux < 0: # only look at mets that are consumed
            #print(met)
            stoich = {met: flux}

            if rxn_id in model.reactions: # biomass test reaction is updated
                rxn = model.reactions.get_by_id(rxn_id)
                rxn.subtract_metabolites(rxn.metabolites)
                rxn.add_metabolites(stoich)
            else: # first time reaction is created
                new_rxn = Reaction(id=rxn_id, name="Biomass reaction", lower_bound=0, upper_bound=1000)
                new_rxn.add_metabolites(stoich)
                model.add_reactions([new_rxn])

            model.objective = rxn_id

            with model:
                change_medium(model, medium_dict)
                try:
                    pfba_flux = pfba(model)
                    if pfba_flux[rxn_id] == 0:
                        sad_mets.append(met.id)
                except Infeasible:
                    print("Cannot get result because pfba is infeasible")

    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        model.remove_reactions([rxn])

    print(sad_mets)

In [ ]:
# this reactions checks the metabolites of reaction the user can specify and checks if the metabolites need can be produced by the model; this is really helpful to investigate pathways and which reaction/metabolite is responsible for the lack of growth in a model
#NOTE: this test is depending on the medium that you use because different media have different compounds that are maybe necessary to produce biomass
def check_early_biomass_component(model, medium_dict, check_rxn_id):
    rxn_id = "objective_check"
    sad_mets = []

    check_mets = model.reactions.get_by_id(check_rxn_id).metabolites
    #check_mets = model.reactions.check_rxn_id.metabolites
    for met, flux in check_mets.items():
        if flux < 0: # only look at mets that are consumed
            #print(met)
            stoich = {met: flux}

            if rxn_id in model.reactions: # biomass test reaction is updated
                rxn = model.reactions.get_by_id(rxn_id)
                rxn.subtract_metabolites(rxn.metabolites)
                rxn.add_metabolites(stoich)
            else: # first time reaction is created
                new_rxn = Reaction(id=rxn_id, name="objective reaction", lower_bound=0, upper_bound=1000)
                new_rxn.add_metabolites(stoich)
                model.add_reactions([new_rxn])

            model.objective = rxn_id

            with model:
                change_medium(model, medium_dict)
                try:
                    pfba_flux = pfba(model)
                    if pfba_flux[rxn_id] == 0:
                        sad_mets.append(met.id)
                except Infeasible:
                    print("Cannot get result because pfba is infeasible")

    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        model.remove_reactions([rxn])

    print(sad_mets)

In [ ]:
maize_root_medium = pd.read_csv("../Datasets/Medium/MRM_whole.csv", sep=";", quotechar='"')
# maize root medium = MRM

In [ ]:
for model in models_curation.values():
    print(model.id)
    check_every_biomass_component(model, maize_root_medium)
    print("-------------")

If there are metabolite names printed that mean that the model is not able to produce all compounds necessary for the objective function by themselves. Below is an investigation of these metabolites.

## kdo2lipid4 (AA5 - Cin)
C06025 (KEGG ID)

In [ ]:
AA5.metabolites.kdo2lipid4_c

In [ ]:
AA5.metabolites.kdo2lipid4_p

In [ ]:
check_early_biomass_component(AA5, maize_root_medium, "UAGAAT")

In [ ]:
get_met(AA5, "kdolipid4_c")

In [ ]:
get_met(AA5, "kdolipid4_c")

## arginine (AA5 - Cin)

In [ ]:
get_met(AA5, "arg__L_c")

In [ ]:
get_met(AA5, "argsuc_c")

In [ ]:
get_met(AA5, "citr__L_c") # is synthesised from orn

In [ ]:
check_early_biomass_component(AA5, maize_root_medium, "ARGSL") # argsuc from citr

In [ ]:
check_early_biomass_component(AA5, maize_root_medium, "ARGSS")

In [ ]:
check_early_biomass_component(AA5, maize_root_medium, "OCBT")

In [ ]:
check_early_biomass_component(AA5, maize_root_medium, "ACOTA")

## Introduce changes to Cin (AA5)
I should add PGAMT and G1PACT to AA5 \
https://biocyc.org/reaction?orgid=GCF_002025665&id=2.3.1.157-RXN this is G1PACT \
https://biocyc.org/reaction?orgid=GCF_002025665&id=5.4.2.10-RXN this is PGAMT \
according to Metacyc both should be in AA5

In [ ]:
new_rxn = copy.deepcopy(AA1.reactions.G1PACT)
AA5.add_reactions([new_rxn])

new_rxn = copy.deepcopy(AA1.reactions.PGAMT)
AA5.add_reactions([new_rxn])

I am adding the ACGS reaction to Cin as this fixes the arginine probleme \
https://biocyc.org/reaction?orgid=GCF_002025665&id=N-ACETYLTRANSFER-RXN#SUMMARY

In [ ]:
new_rxn = copy.deepcopy(AA1.reactions.ACGS)
AA5.add_reactions([new_rxn])

In [ ]:
AA5.objective = "Growth"

In [ ]:
check_early_biomass_component(AA5, maize_root_medium, "Growth")

## nad (AA1 - Sma)

In [ ]:
get_met(AA1, "nad_c")

In [ ]:
nad_rxns = []
for rxn in AA1.reactions:
    if AA1.metabolites.nad_c in rxn.metabolites.keys() and AA1.metabolites.nadh_c not in rxn.metabolites.keys():
        print(rxn.id)
        nad_rxns.append(rxn.id)
# get reactions where only nad is used and not nadh to prevent reactions where we only have recycling

In [ ]:
for rxn in nad_rxns:
    get_rxn(AA1, rxn, charge=False)
    print("-------")

only NADS1, NADS2 and NMNAT produce NAD


In [ ]:
get_met(AA1, "dnad_c")

NADS2 6.3.5.1 \
NNATr 2.7.7.18

In [ ]:
get_met(AA1, "nicrnt_c")

NNDPR 2.4.2.19 - this reaction is missing in AA1 but present in the rest of the models \
https://biocyc.org/pathway?orgid=GCF_000072485&id=NADSYN-PWY \

In [ ]:
get_rxn_unknown(models_curation, "NNDPR")

QULNS 2.5.1.72

In [ ]:
get_rxn_unknown(models_curation, "QULNS")

## nadp (Sma - AA1)

In [ ]:
nadp_rxns = []
for rxn in AA1.reactions:
    if AA1.metabolites.nadp_c in rxn.metabolites.keys() and AA1.metabolites.nadph_c not in rxn.metabolites.keys():
        print(rxn.id)
        nadp_rxns.append(rxn.id)
# get reactions where only nadp is used and not nadph to prevent reactions where we only have recycling

In [ ]:
for rxn in nadp_rxns:
    get_rxn(AA1, rxn, charge=False)
    print("-------")

as soon as we can produce NAD, we can also produce NADP

## Introduce changes to Sma (AA1)

fixes for nad

In [ ]:
new_met = copy.deepcopy(AA2.metabolites.iasp_c)
AA1.add_metabolites([new_met])
new_met = copy.deepcopy(AA2.metabolites.quln_c)
AA1.add_metabolites([new_met])
new_rxn = copy.deepcopy(AA2.reactions.NNDPR)
AA1.add_reactions([new_rxn])
new_rxn = copy.deepcopy(AA2.reactions.QULNS)
AA1.add_reactions([new_rxn])
new_rxn = copy.deepcopy(AA2.reactions.ASPO6)
AA1.add_reactions([new_rxn])

fixes for sheme

In [ ]:
# sheme fixes (i.e. we remove it as there is no evidence it should be in Sma)
growth_rxn = AA1.reactions.get_by_id("Growth")
met = AA1.metabolites.get_by_id("sheme_c")

if met in growth_rxn.metabolites:
    growth_rxn.subtract_metabolites({met: growth_rxn.metabolites[met]})

In [ ]:
sheme_rxns = ["SHEME2abcpp", "SHEMEtonex", "EX_sheme_e"]

for rxn_id in sheme_rxns:
    if rxn_id in AA1.reactions:
        rxn = AA1.reactions.get_by_id(rxn_id)
        AA1.remove_reactions([rxn])

In [ ]:
sheme_ids = ["sheme_c", "sheme_e", "sheme_p"]

for met_id in sheme_ids:
    if met_id in AA1.metabolites:
        met = AA1.metabolites.get_by_id(met_id)
        AA1.metabolites.remove(met)

fixes for proline

In [ ]:
new_rxn = copy.deepcopy(AA2.reactions.G5SADs)
AA1.add_reactions([new_rxn])

write model

In [ ]:
AA1.objective = "Growth"
write_sbml_model(AA1, "../Models/02_afterCuration/AA1.xml")

In [ ]:
check_early_biomass_component(AA1, maize_root_medium, "Growth")

## thm (AA2 - Bpi)

In [ ]:
get_met(AA2, "thm_c")

In [ ]:
get_met(AA2, "thmpp_c")

In [ ]:
get_rxn_unknown(models_curation, "THMP")

fixes

In [ ]:
new_rxn = copy.deepcopy(AA7.reactions.THMP)
AA2.add_reactions([new_rxn])

In [ ]:
write_sbml_model(AA2, "../Models/02_afterCuration/AA2.xml")

# Benzoate & Menaquinol
We need to remove mql8 from the biomass for AA7, AA6 and AA1, AA2. \
Some models or rather the bacteria are lacking the synthesis pathway for mql8 because the pathway originated from benzoate (bz) is not really legit because it is only present in one bigg model (iAF987) without any cross references to other databases. \
That means, there is currently no info how AA1/2/6/7 synthesise mql8.

In [ ]:
fix_models = ["AA1","AA2", "AA6", "AA7"]
for model in models_curation.values():
    if model.id in fix_models:
        growth_rxn = model.reactions.get_by_id("Growth")
        met = model.metabolites.get_by_id("mql8_c")

        if met in growth_rxn.metabolites:
            growth_rxn.subtract_metabolites({met: growth_rxn.metabolites[met]})

# Ubiquinol

In [ ]:
# 2.5.1.39
get_rxn_unknown(models_curation, "HBZOPT")
# missing in AA3 but that is ok because gram positive and using menaquinol

In [ ]:
# 4.1.1.98
get_rxn_unknown(models_curation, "OPHBDC")

In [ ]:
# 1.14.13.240
get_rxn_unknown(models_curation, "OPHHX")

In [ ]:
# these are missing reactions to go from 2ohph (3-(all-trans-octaprenyl)benzene-1,2-diol to q8h2 (ubiquinol-8)
# 2.1.1.222 = OHPHM = 2ohph_c + amet_c → 2omph_c + ahcys_c + h_c (2ohph_c we have in our models)
# 1.14.13.- = OMPHHX = 2omph_c + 0.5 o2_c → 2ombzl_c
# 2.1.1.201 = OMBZLM = 2ombzl_c + amet_c → 2ommbl_c + ahcys_c + h_c
# 1.14.99.60 = OMMBLHX = 2ommbl_c + 0.5 o2_c → 2omhmbl_c
# 2.1.1.64 = DMQMT = 2omhmbl_c + amet_c → ahcys_c + h_c + q8h2_c

# beforehand we need to add the missing metabolites
# 2-methoxy-6-(all-trans-octaprenyl)phenol = 2omph_c
# 2-methoxy-6-(all-trans-octaprenyl)-1,4-benzoquinol = 2ombzl_c
# 6-methoxy-3-methyl-2-all-trans-octaprenyl-1,4-benzoquinol = 2ommbl_c
# 3-demethylubiquinol-8 = 2omhmbl_c

# we also need to check if all the other side metabolites from these reactions are present
# also check BIGG if these metabolites exist in general and then we can introduce them into our models using BIGG IDs

In [ ]:
def add_new_rxn(model, id, name, lb, ub, stoich):
    if id not in model.reactions:
        new_rxn = Reaction(id=id, name=name, lower_bound=lb, upper_bound=ub)

        met_objs = {}
        for met_id, coeff in stoich.items():
            met_obj = model.metabolites.get_by_id(met_id) if met_id in model.metabolites else None
            if met_obj is None:
                return f"{met_id} not in {model.id}; {id} was not added"
            met_objs[met_obj] = coeff

        new_rxn.add_metabolites(met_objs)
        model.add_reactions([new_rxn])
        return None
    else:
        return f"{id} already in {model.id}"

In [ ]:
def add_new_met(model, id, name, formula, charge, compartment):
    if id not in model.metabolites:
        new_met = Metabolite(id, name=name, formula=formula, charge=charge, compartment=compartment)
        model.add_metabolites([new_met])
        return None
    else:
        return f"{id} already in {model.id}"

In [ ]:
def add_ubi_metabolites(model):
    add_new_met(model, "2omph_c",'2-Octaprenyl-6-methoxyphenol', 'C47H72O2', 0, "C_c")
    add_new_met(model, "2ombzl_c",'2-Octaprenyl-6-methoxy-1,4-benzoquinol', 'C47H72O3', 0, "C_c")
    add_new_met(model, "2ommbl_c",'2-Octaprenyl-3-methyl-6-methoxy-1,4-benzoquinol', 'C48H74O3', 0, "C_c")
    add_new_met(model, "2omhmbl_c",'2-Octaprenyl-3-methyl-5-hydroxy-6-methoxy-1,4-benzoquinol', 'C48H74O4', 0, "C_c")

def add_OHPHM(model):
    add_new_rxn(model, "OHPHM", "2-octaprenyl-6-hydroxyphenol methylase", 0, 1000,
                {'2ohph_c': -1.0, '2omph_c': 1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0})

def add_OMPHHX(model):
    add_new_rxn(model, "OMPHHX", "2-octaprenyl-6-methoxyphenol hydroxylase", 0, 1000,
                {'2ombzl_c': 1.0, '2omph_c': -1.0, 'o2_c': -0.5})

def add_OMBZLM(model):
    add_new_rxn(model, "OMBZLM", "2-Octaprenyl-6-methoxy-benzoquinol methylase", 0, 1000,
                {'2ombzl_c': -1.0, '2ommbl_c': 1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0})

def add_OMMBLHX(model):
    add_new_rxn(model, "OMMBLHX", "2-Octaprenyl-3-methyl-6-methoxy-1,4-benzoquinol hydroxylase", 0, 1000,
                {'2omhmbl_c': 1.0, '2ommbl_c': -1.0, 'o2_c': -0.5})

def add_DMQMT(model):
    add_new_rxn(model, "DMQMT", "3-Dimethylubiquinonol 3-methyltransferase", 0, 1000,
                {'2omhmbl_c': -1.0, 'ahcys_c': 1.0, 'amet_c': -1.0, 'h_c': 1.0, 'q8h2_c': 1.0})


In [ ]:
models_ubiquinol = ["AA1", "AA2", "AA6", "AA7"]
for model in models_curation.values():
    if model.id in models_ubiquinol:
        add_ubi_metabolites(model)
        add_OHPHM(model)
        add_OMPHHX(model)
        add_OMBZLM(model)
        add_OMMBLHX(model)
        add_DMQMT(model)

In [ ]:
# add Ubiquinol to Biomass for those 4 instead of Menaquinol which we already kicked out
def add_met_to_rxn(model, rxn_id, met_id, coeff):
    rxn = model.reactions.get_by_id(rxn_id)
    met = model.metabolites.get_by_id(met_id)

    if met not in rxn.metabolites:
        rxn.add_metabolites({met: coeff})
        return None
    else:
        return f"{met_id} already in {rxn_id}"

for model in models_curation.values():
    if model.id in models_ubiquinol:
        add_met_to_rxn(model, "Growth", "q8h2_c", -0.0001)

# Sucrose Fixes
Based on our in vitro results, we investigated the pathways related to sucrose as in silico growth assays differed largely from the in vitro results; based on KEGG and mMetacyc our models contain too many sucrose related reactions.

In [ ]:
def delete_reaction(rxn_id, model):
    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        all_mets = rxn.metabolites.keys()
        model.remove_reactions([rxn])

        # check for orphans
        for met in all_mets:
            if len(met.reactions) == 1:
                print(f"you created orphan metabolite that has only 1 reaction: {met}")

    else:
        print(f"reaction {rxn_id} not in model {model.id}")


def delete_metabolite(model, met_id):
    if met_id in model.metabolites:
        if len(model.metabolites.get_by_id(met_id).reactions) == 0:
            met = model.metabolites.get_by_id(met_id)
            model.metabolites.remove(met)

        else:
            print(f'metabolite {met_id} cannot be deleted from {model.id} because of reaction(s): {model.metabolites.get_by_id(met_id).reactions}')
    else:
        print(f"metabolite {met_id} does not exist in {model.id}")

In [ ]:
def del_SUCpts(model):
    rxn_id = "SUCpts"
    delete_reaction(rxn_id, model)

def del_SUCRt2(model):
    rxn_id = "SUCRt2"
    delete_reaction(rxn_id, model)

def del_SUCR(model):
    rxn_id = "SUCR"
    delete_reaction(rxn_id, model)

def del_HEX7(model):
    rxn_id = "HEX7"
    delete_reaction(rxn_id, model)

def del_EX_sucr(model):
    rxn_id = "EX_sucr_e"
    delete_reaction(rxn_id, model)

def del_SPP(model):
    rxn_id = "SPP"
    delete_reaction(rxn_id, model)

def del_FFSD(model):
    rxn_id = "FFSD"
    delete_reaction(rxn_id, model)

def del_sucr_e(model):
    met_id = "sucr_e"
    delete_metabolite(model, met_id)

def del_sucr_c(model):
    met_id = "sucr_c"
    delete_metabolite(model, met_id)

def del_fru_c(model):
    met_id = "fru_c"
    delete_metabolite(model, met_id)

def del_suc6p_c(model):
    met_id = "suc6p_c"
    delete_metabolite(model, met_id)

In [ ]:
# AA1
del_SUCpts(AA1)
del_EX_sucr(AA1)
del_SPP(AA1)
del_SUCR(AA1)
del_sucr_e(AA1)
del_suc6p_c(AA1)
del_sucr_c(AA1)

# AA2
del_SUCpts(AA2)
del_SUCRt2(AA2)

# AA3
del_SUCpts(AA3)
del_SUCRt2(AA3)
del_EX_sucr(AA3)
del_FFSD(AA3)
del_sucr_e(AA3)
del_suc6p_c(AA3)

# AA5
del_SUCRt2(AA5)
del_SUCR(AA5)
del_HEX7(AA5)
del_EX_sucr(AA5)
del_sucr_e(AA5)
del_sucr_c(AA5)
del_fru_c(AA5)

# AA6
del_SUCpts(AA6)
del_EX_sucr(AA6)
del_FFSD(AA6)
del_sucr_e(AA6)
del_suc6p_c(AA6)


# Final save after curation

In [ ]:
# save all curated model as file
for model_name, model in models_curation.items():
    path = f"../Models/02_afterCuration/{model_name[:3]}.xml"
    write_sbml_model(model, path)

# Investigate auxotrophy of AA2 (Bpi)

In [ ]:
for model in models_curation.values():
    growth_val = test_medium(model, maize_root_medium)
    print(model.id, ": ", growth_val)

In [ ]:
AA2_requirements = list(minimal_medium(AA2, 5).index)
# minimal medium function of cobrapy creates a medium with minimal amount of components to produce biomass

In [ ]:
# all components from the minimal medium of cobrapy are added one at the time to the maize root medium
create_medium(AA2_requirements, list(maize_root_medium["reaction"]), models_curation, 1000)